# Agentic Codebase Runtime v15 — protocol-compiled runtime, braided memory, patch election, and one huge main block

This version advances the runtime with:

- hierarchical plan election across candidate task packs and strategy genomes
- trust-weighted ColorSync mentorship and rare-condition palette replay
- counterfactual rollout scoring before execution
- explanation graph + operator alerts + champion archive artifacts
- a single explicit `main()` entrypoint that runs election, arena episodes, verification, repairs, and artifact export

It remains notebook-first, Starling-ready, MAPF-aware, chromatic, and PennyLane-ready.


In [ ]:
# @title 1) Optional pip installs for local notebook use
RUN_INSTALLS = False  # set True in Colab / notebook environment when you want installs

if RUN_INSTALLS:
    print("Installing planner dependencies...")
    get_ipython().system("pip -q install pennylane psutil nltk summa")
else:
    print("RUN_INSTALLS=False -> skipped pip install cell.")


In [ ]:
# @title 2) Optional GPU check + llama-cpp-python CUDA install + Starling GGUF download
import os
import shutil
from pathlib import Path

RUN_GPU_SETUP = False   # set True when you want the CUDA llama-cpp install
RUN_MODEL_DOWNLOAD = False  # set True to download the GGUF
MODEL_DIR = "/content/models"
if not os.access("/content", os.W_OK):
    MODEL_DIR = "/mnt/data/models"
MODEL_NAME = "starling-lm-7b-alpha.Q4_K_M.gguf"
GGUF_PATH = os.path.join(MODEL_DIR, MODEL_NAME)
GGUF_URL = "https://huggingface.co/TheBloke/Starling-LM-7B-alpha-GGUF/resolve/main/starling-lm-7b-alpha.Q4_K_M.gguf"
GGUF_SHA256 = "0951cbc1a6c3ed8d081db59366ccccf09ed52a4cfd5191812665b911fe6c669a"

print("GPU check:")
if shutil.which("nvidia-smi"):
    get_ipython().system("nvidia-smi")
else:
    print("nvidia-smi not found in this environment.")

if RUN_GPU_SETUP:
    print("Installing llama-cpp-python CUDA wheel...")
    get_ipython().system("pip install -U llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 --no-cache-dir")
else:
    print("RUN_GPU_SETUP=False -> skipped llama-cpp-python CUDA install.")

try:
    from llama_cpp import Llama
    import llama_cpp
    print(f"llama-cpp-python detected: {llama_cpp.__version__}")
except Exception as e:
    print(f"llama-cpp-python not ready in this environment: {e}")

os.makedirs(MODEL_DIR, exist_ok=True)
if RUN_MODEL_DOWNLOAD:
    if Path(GGUF_PATH).exists():
        print("Model already present, skipping download.")
    else:
        get_ipython().system(f'wget --show-progress --progress=bar:force:noscroll -O "{GGUF_PATH}" "{GGUF_URL}"')
    print("Verifying SHA256...")
    get_ipython().system(f'echo "{GGUF_SHA256}  {GGUF_PATH}" | sha256sum -c -')
else:
    print("RUN_MODEL_DOWNLOAD=False -> skipped GGUF download.")

os.environ["STARLING_GGUF_PATH"] = GGUF_PATH
print("STARLING_GGUF_PATH =", os.environ["STARLING_GGUF_PATH"])


In [ ]:
# @title 3) Imports, configuration, and local model helpers
from __future__ import annotations

import ast
import compileall
import hashlib
import json
import math
import os
import random
import re
import shutil
import sqlite3
import statistics
import subprocess
import sys
import textwrap
import time
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import psutil

try:
    import pennylane as qml
except Exception:
    qml = None

try:
    from llama_cpp import Llama
except Exception:
    Llama = None

try:
    from summa import summarizer as summa_summarizer
    from summa import keywords as summa_keywords
except Exception:
    summa_summarizer = None
    summa_keywords = None

try:
    import nltk
    from nltk.tokenize import sent_tokenize as nltk_sent_tokenize
except Exception:
    nltk = None
    nltk_sent_tokenize = None

if nltk is not None:
    try:
        nltk.download("punkt", quiet=True)
    except Exception:
        pass

# -------------------------
# Runtime switches
# -------------------------
DEBUG_VERBOSE = False
ENABLE_STARLING_LLM = False   # set True when llama-cpp-python and GGUF are available
USE_SQLITE_MEMORY = True
SEED = 7
random.seed(SEED)
np.random.seed(SEED)

# -------------------------
# Starling / llama.cpp settings
# -------------------------
STARLING_GGUF_PATH = os.environ.get("STARLING_GGUF_PATH", "")
STARLING_CTX_SIZE = 4096
STARLING_THREADS = max(4, min(8, os.cpu_count() or 4))
STARLING_N_GPU_LAYERS = 0
STARLING_MAX_TOKENS = 1400
STARLING_TEMPERATURE = 0.33

# -------------------------
# Runtime output paths
# -------------------------
WORKSPACE_ROOT = Path("/tmp/agentic_workspace_v10")
MEMORY_DB_PATH = WORKSPACE_ROOT / "agentic_codebase_memory_v10.db"
RUNTIME_BUNDLE_PATH = WORKSPACE_ROOT / "docs" / "runtime_bundle.json"

# -------------------------
# Project objective
# -------------------------
DEFAULT_SPEC = """
Build a notebook-first agentic codebase runtime with a Python backend, optional React dashboard,
SQLite memory, MAPF-aware scheduling, chromatic agent cognition, PennyLane-ready quantum scoring,
verification gates, repair loops, ColorSync cross-agent learning, evolutionary plan election,
and local Starling LLM task generation support.
""".strip()

COLOR_ORDER = ["gold", "emerald", "amber", "violet", "crimson", "white", "cyan"]
COLOR_INDEX = {name: idx for idx, name in enumerate(COLOR_ORDER)}
LANE_ORDER = ["architecture", "backend", "frontend", "infra", "test", "docs"]
SCHEDULER_MODE_ORDER = ["maintenance", "reasoning", "anomaly_detection", "delivery"]

CONCEPT_FILES = [
    Path("/mnt/data/agentic_codebase_blog.md"),
    Path("/mnt/data/agentic_codebase_blog_payload.json"),
]

def debug_print(tag: str, message: str) -> None:
    if DEBUG_VERBOSE:
        print(f"[{tag}] {message}")

def stable_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]

def clamp(x: float, lo: float = 0.0, hi: float = 1.0) -> float:
    return max(lo, min(hi, float(x)))

def normalize_dict(values: Dict[str, float]) -> Dict[str, float]:
    total = float(sum(max(0.0, v) for v in values.values())) or 1.0
    return {k: round(max(0.0, float(v)) / total, 6) for k, v in values.items()}

def extract_json_block(text: str) -> Dict[str, Any]:
    text = (text or "").strip()
    if not text:
        return {}
    fences = re.findall(r"```json\s*(.*?)```", text, flags=re.S | re.I)
    candidates = fences + re.findall(r"(\{.*\})", text, flags=re.S)
    for candidate in candidates:
        try:
            return json.loads(candidate)
        except Exception:
            continue
    return {}

def trim_prompt(text: str, limit_chars: int = 16000) -> str:
    text = text.strip()
    if len(text) <= limit_chars:
        return text
    half = limit_chars // 2
    return text[:half] + "\n[... trimmed ...]\n" + text[-half:]

def summarize_text_locally(text: str, ratio: float = 0.22, max_sentences: int = 6) -> str:
    if not text.strip():
        return ""
    if summa_summarizer is not None:
        try:
            out = summa_summarizer.summarize(text, ratio=ratio)
            if out.strip():
                return out.strip()
        except Exception:
            pass
    if nltk_sent_tokenize is not None:
        try:
            sents = nltk_sent_tokenize(text)
            return " ".join(sents[:max_sentences])
        except Exception:
            pass
    return text[:1600]

def keyword_extract_locally(text: str, limit: int = 18) -> List[str]:
    if not text.strip():
        return []
    if summa_keywords is not None:
        try:
            raw = summa_keywords.keywords(text, words=limit, split=True)
            return [x.strip() for x in raw if x.strip()][:limit]
        except Exception:
            pass
    words = re.findall(r"[A-Za-z][A-Za-z0-9_\-]{3,}", text.lower())
    uniq = []
    for w in words:
        if w not in uniq:
            uniq.append(w)
    return uniq[:limit]

LLM = None

def load_llm() -> Optional[Any]:
    global LLM
    if not ENABLE_STARLING_LLM:
        return None
    if LLM is not None:
        return LLM
    if Llama is None:
        debug_print("llm", "llama_cpp unavailable")
        return None
    if not STARLING_GGUF_PATH or not os.path.exists(STARLING_GGUF_PATH):
        debug_print("llm", f"GGUF not found at {STARLING_GGUF_PATH}")
        return None
    LLM = Llama(
        model_path=STARLING_GGUF_PATH,
        n_ctx=int(STARLING_CTX_SIZE),
        n_threads=int(STARLING_THREADS),
        n_gpu_layers=int(STARLING_N_GPU_LAYERS),
        verbose=False,
    )
    return LLM

def llm_complete(prompt: str) -> str:
    model = load_llm()
    if model is None:
        return ""
    out = model(
        trim_prompt(prompt),
        max_tokens=int(STARLING_MAX_TOKENS),
        temperature=float(STARLING_TEMPERATURE),
        top_p=0.92,
        repeat_penalty=1.08,
        stop=["</end>", "\n\n\n\n"],
    )
    return out["choices"][0]["text"].strip()

def runtime_snapshot() -> Dict[str, Any]:
    mem = psutil.virtual_memory()
    return {
        "python": sys.version.split()[0],
        "cpu_count": os.cpu_count(),
        "memory_total_gb": round(mem.total / (1024**3), 2),
        "memory_available_gb": round(mem.available / (1024**3), 2),
        "pennylane_available": qml is not None,
        "llama_cpp_available": Llama is not None,
        "starling_enabled": ENABLE_STARLING_LLM,
        "gguf_path": STARLING_GGUF_PATH,
    }

runtime_snapshot()


In [ ]:
# @title 4) Core planner state, memory, deterministic tasks, chromatic agents, and concept-bank loading
@dataclass
class ColorVector:
    values: np.ndarray

    @staticmethod
    def from_dict(d: Optional[Dict[str, float]] = None, default_bias: Optional[Dict[str, float]] = None) -> "ColorVector":
        base = {name: 0.0 for name in COLOR_ORDER}
        if default_bias:
            base.update({k: float(v) for k, v in default_bias.items() if k in base})
        if d:
            base.update({k: float(v) for k, v in d.items() if k in base})
        arr = np.array([max(0.0, float(base[name])) for name in COLOR_ORDER], dtype=float)
        total = float(arr.sum()) or 1.0
        return ColorVector(arr / total)

    @staticmethod
    def random(seed: Optional[int] = None) -> "ColorVector":
        rng = np.random.default_rng(seed)
        arr = rng.random(len(COLOR_ORDER))
        return ColorVector(arr / float(arr.sum()))

    def to_dict(self) -> Dict[str, float]:
        total = float(self.values.sum()) or 1.0
        return {name: round(float(self.values[idx] / total), 6) for idx, name in enumerate(COLOR_ORDER)}

    def blend(self, other: "ColorVector", alpha: float = 0.5) -> "ColorVector":
        arr = ((1.0 - alpha) * self.values) + (alpha * other.values)
        total = float(arr.sum()) or 1.0
        return ColorVector(arr / total)

    def similarity(self, other: "ColorVector") -> float:
        return float(np.dot(self.values, other.values))

@dataclass
class TaskNode:
    task_id: str
    title: str
    lane: str
    description: str
    dependencies: List[str] = field(default_factory=list)
    resources: List[str] = field(default_factory=list)
    status: str = "pending"
    priority: float = 0.5
    reversibility: float = 0.7
    uncertainty: float = 0.35
    source: str = "base"
    color_hint: Dict[str, float] = field(default_factory=dict)
    rationale: str = ""
    rare_condition: bool = False

@dataclass
class AgentProfile:
    name: str
    specialty: str
    lanes: List[str]
    max_parallel_tasks: int = 1

@dataclass
class Reservation:
    resource: str
    task_id: str
    agent_name: str
    start_step: int
    end_step: int

@dataclass
class PaletteBroadcast:
    source_agent: str
    task_id: str
    confidence: float
    palette: Dict[str, float]
    rare_condition: bool
    note: str

@dataclass
class ChromaticAgentState:
    profile: AgentProfile
    palette: ColorVector
    trust: Dict[str, float] = field(default_factory=dict)
    completed: List[str] = field(default_factory=list)

    def lane_match(self, task: TaskNode) -> float:
        return 1.0 if task.lane in self.profile.lanes else 0.45

@dataclass
class PlannerState:
    spec: str
    tasks: Dict[str, TaskNode] = field(default_factory=dict)
    reservations: List[Reservation] = field(default_factory=list)
    completed: List[str] = field(default_factory=list)
    failed: List[str] = field(default_factory=list)
    oversight: Dict[str, float] = field(default_factory=dict)
    broadcasts: List[PaletteBroadcast] = field(default_factory=list)

    def add_task(self, task: TaskNode) -> None:
        self.tasks[task.task_id] = task

    def add_tasks(self, tasks: Iterable[TaskNode]) -> None:
        for task in tasks:
            self.add_task(task)

    def ready_tasks(self) -> List[TaskNode]:
        items = []
        for task in self.tasks.values():
            if task.status != "pending":
                continue
            if all(dep in self.completed for dep in task.dependencies):
                items.append(task)
        return sorted(items, key=lambda t: (-t.priority, t.uncertainty, t.task_id))

class MemoryStore:
    def __init__(self, db_path: Path):
        self.db_path = db_path
        db_path.parent.mkdir(parents=True, exist_ok=True)
        self.conn = sqlite3.connect(str(db_path))
        self.conn.execute(
            """
            CREATE TABLE IF NOT EXISTS traces (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                created_at REAL,
                phase TEXT,
                task_id TEXT,
                lane TEXT,
                event_type TEXT,
                payload TEXT
            )
            """
        )
        self.conn.commit()

    def write_trace(self, phase: str, task: Optional[TaskNode], event_type: str, payload: Dict[str, Any]) -> None:
        self.conn.execute(
            "INSERT INTO traces(created_at, phase, task_id, lane, event_type, payload) VALUES (?, ?, ?, ?, ?, ?)",
            (
                time.time(),
                phase,
                task.task_id if task else "",
                task.lane if task else "",
                event_type,
                json.dumps(payload, sort_keys=True),
            ),
        )
        self.conn.commit()

    def recent_traces(self, limit: int = 40) -> List[Dict[str, Any]]:
        cur = self.conn.execute(
            "SELECT created_at, phase, task_id, lane, event_type, payload FROM traces ORDER BY id DESC LIMIT ?",
            (int(limit),),
        )
        rows = cur.fetchall()
        out = []
        for created_at, phase, task_id, lane, event_type, payload in rows:
            out.append(
                {
                    "created_at": created_at,
                    "phase": phase,
                    "task_id": task_id,
                    "lane": lane,
                    "event_type": event_type,
                    "payload": json.loads(payload or "{}"),
                }
            )
        return out

    def digest(self, limit: int = 40) -> str:
        traces = self.recent_traces(limit=limit)
        lines = []
        for item in traces:
            payload = item.get("payload", {})
            note = payload.get("note") or payload.get("detail") or payload.get("summary") or json.dumps(payload)[:180]
            lines.append(f"[{item['phase']}/{item['event_type']}] {item['task_id']} {note}")
        return "\n".join(lines[:limit])

LANE_RULES = {
    "architecture": ["architecture", "system", "design", "plan", "api", "multi-agent", "palette", "quantum"],
    "backend": ["backend", "server", "database", "auth", "service", "endpoint", "python", "memory"],
    "frontend": ["frontend", "ui", "ux", "dashboard", "client", "react"],
    "infra": ["infra", "ci", "deploy", "docker", "pipeline", "monitoring", "gpu"],
    "test": ["test", "qa", "verification", "lint", "contract", "smoke"],
    "docs": ["docs", "documentation", "readme", "guide", "notebook", "operator"],
}

DEFAULT_AGENT_POOL = [
    AgentProfile(name="Planner Agent", specialty="planning", lanes=["architecture", "docs"]),
    AgentProfile(name="Backend Agent", specialty="backend", lanes=["backend", "architecture"]),
    AgentProfile(name="Frontend Agent", specialty="frontend", lanes=["frontend", "docs"]),
    AgentProfile(name="Infra Agent", specialty="infra", lanes=["infra", "backend"]),
    AgentProfile(name="Verifier Agent", specialty="verification", lanes=["test", "docs", "backend"]),
    AgentProfile(name="Repair Agent", specialty="repair", lanes=["backend", "test", "infra"]),
]

DEFAULT_AGENT_BIAS = {
    "Planner Agent": {"gold": 0.24, "cyan": 0.18, "violet": 0.18, "white": 0.12, "emerald": 0.12, "amber": 0.10, "crimson": 0.06},
    "Backend Agent": {"emerald": 0.22, "gold": 0.18, "amber": 0.16, "white": 0.14, "cyan": 0.11, "violet": 0.10, "crimson": 0.09},
    "Frontend Agent": {"cyan": 0.22, "gold": 0.20, "violet": 0.17, "white": 0.13, "emerald": 0.11, "amber": 0.10, "crimson": 0.07},
    "Infra Agent": {"amber": 0.20, "emerald": 0.18, "crimson": 0.16, "white": 0.14, "gold": 0.12, "cyan": 0.10, "violet": 0.10},
    "Verifier Agent": {"amber": 0.24, "white": 0.18, "crimson": 0.16, "emerald": 0.15, "gold": 0.10, "violet": 0.09, "cyan": 0.08},
    "Repair Agent": {"crimson": 0.21, "white": 0.20, "amber": 0.16, "emerald": 0.14, "gold": 0.11, "violet": 0.10, "cyan": 0.08},
}

def infer_lanes(spec: str) -> List[str]:
    lower = spec.lower()
    chosen = []
    for lane, keywords in LANE_RULES.items():
        if any(word in lower for word in keywords):
            chosen.append(lane)
    return chosen or ["architecture", "backend", "test", "docs"]

def deterministic_seed_tasks(spec: str) -> List[TaskNode]:
    tasks = [
        TaskNode(
            task_id="T001",
            title="Author runtime architecture contract",
            lane="architecture",
            description="Define the runtime phases, state transitions, artifacts, and contracts for the notebook-first planner.",
            resources=["contracts/runtime_architecture.json", "docs/runtime_architecture.md"],
            priority=0.96,
            uncertainty=0.16,
            color_hint={"gold": 0.25, "white": 0.18, "emerald": 0.14, "cyan": 0.16},
            rationale="The runtime needs a stable control surface before deeper automation.",
        ),
        TaskNode(
            task_id="T002",
            title="Implement chromatic quantum circuit pack",
            lane="backend",
            description="Provide intent, task pigment, scheduler, memory, and reset evaluators for scoring tasks and system state.",
            dependencies=["T001"],
            resources=["src/runtime/quantum.py", "tests/test_quantum_pack.py"],
            priority=0.94,
            uncertainty=0.22,
            color_hint={"gold": 0.20, "violet": 0.18, "cyan": 0.14, "emerald": 0.16, "amber": 0.12},
            rationale="The quantum pack drives nonlinear scoring and scheduler mode selection.",
        ),
        TaskNode(
            task_id="T003",
            title="Build MAPF-aware scheduler and reservation ledger",
            lane="backend",
            description="Create a resource-aware assignment layer that avoids collisions and encodes time reservations.",
            dependencies=["T001"],
            resources=["src/runtime/scheduler.py", "contracts/reservation_ledger.json"],
            priority=0.92,
            uncertainty=0.21,
            color_hint={"emerald": 0.20, "amber": 0.18, "gold": 0.14, "white": 0.14},
            rationale="Concurrent agents need conflict-aware routing.",
        ),
        TaskNode(
            task_id="T004",
            title="Add SQLite memory fusion and trace schema",
            lane="backend",
            description="Persist task traces, palette broadcasts, resets, and verification outcomes for later reuse.",
            dependencies=["T001"],
            resources=["src/runtime/memory.py", "contracts/memory_trace_schema.json"],
            priority=0.91,
            uncertainty=0.19,
            color_hint={"emerald": 0.18, "white": 0.18, "violet": 0.12, "amber": 0.10},
            rationale="Memory is required for calibration, reflection, and repair selection.",
        ),
        TaskNode(
            task_id="T005",
            title="Create verification and repair harness",
            lane="test",
            description="Run syntax checks, contract checks, and structured repair synthesis after runtime generation.",
            dependencies=["T002", "T003", "T004"],
            resources=["src/runtime/verify.py", "tests/test_runtime_verify.py"],
            priority=0.90,
            uncertainty=0.25,
            color_hint={"amber": 0.22, "white": 0.18, "crimson": 0.12, "emerald": 0.14},
            rationale="The runtime should self-check and recover from drift.",
        ),
        TaskNode(
            task_id="T006",
            title="Document operator workflow and execution contract",
            lane="docs",
            description="Write the operator guide for setup, generation, election, execution, verification, and artifact review.",
            dependencies=["T001", "T005"],
            resources=["README.md", "docs/operator_workflow.md"],
            priority=0.84,
            uncertainty=0.20,
            color_hint={"white": 0.22, "gold": 0.18, "emerald": 0.14},
            rationale="The runtime should be inspectable and usable by a human operator.",
        ),
    ]
    return tasks

def bootstrap_agents() -> List[ChromaticAgentState]:
    out = []
    for profile in DEFAULT_AGENT_POOL:
        palette = ColorVector.from_dict(default_bias=DEFAULT_AGENT_BIAS.get(profile.name, {}))
        trust = {other.name: 0.18 for other in DEFAULT_AGENT_POOL if other.name != profile.name}
        out.append(ChromaticAgentState(profile=profile, palette=palette, trust=trust))
    return out

def load_concept_bank() -> Dict[str, Any]:
    chunks = []
    for path in CONCEPT_FILES:
        if not path.exists():
            continue
        try:
            text = path.read_text(errors="ignore")
            chunks.append(text[:220000])
        except Exception:
            continue
    raw = "\n\n".join(chunks)
    summary = summarize_text_locally(raw, ratio=0.12, max_sentences=10) if raw else ""
    keywords = keyword_extract_locally(raw, limit=28) if raw else []
    headings = re.findall(r"^#+\s+(.+)$", raw, flags=re.M)[:20] if raw else []
    return {
        "raw_size": len(raw),
        "summary": summary,
        "keywords": keywords,
        "headings": headings,
    }

concept_bank = load_concept_bank()
{
    "loaded_bytes": concept_bank["raw_size"],
    "keyword_count": len(concept_bank["keywords"]),
    "sample_keywords": concept_bank["keywords"][:10],
}


In [ ]:
# @title 5) Quantum circuit pack + calibration + advanced Starling task generator
def _normalize_obs(values: Iterable[float]) -> List[float]:
    out = []
    for v in values:
        v = float(v)
        out.append(round((v + 1.0) / 2.0, 6))
    return out

class QuantumCircuitPack:
    def __init__(self):
        self.backend = "pennylane" if qml is not None else "classical_surrogate"
        if qml is not None:
            self._init_qnodes()

    def _init_qnodes(self):
        intent_device = qml.device("default.qubit", wires=4)
        task_device = qml.device("default.qubit", wires=4)
        scheduler_device = qml.device("default.qubit", wires=4)
        memory_device = qml.device("default.qubit", wires=4)
        reset_device = qml.device("default.qubit", wires=4)

        @qml.qnode(intent_device)
        def intent_qnode(features):
            for i, value in enumerate(features):
                qml.RY(float(value) * np.pi, wires=i)
            qml.CNOT(wires=[0, 1]); qml.CNOT(wires=[1, 2]); qml.CNOT(wires=[2, 3])
            qml.RZ(float(np.mean(features)) * np.pi, wires=0)
            qml.RX(float(np.std(features)) * np.pi, wires=2)
            return [qml.expval(qml.PauliZ(i)) for i in range(4)]

        @qml.qnode(task_device)
        def task_qnode(features):
            for i, value in enumerate(features):
                qml.RY(float(value) * np.pi, wires=i)
                qml.RX(float(1.0 - value) * np.pi / 2.0, wires=i)
            qml.CNOT(wires=[0, 2]); qml.CNOT(wires=[1, 3])
            qml.RZ(float(np.mean(features)) * np.pi / 2.0, wires=1)
            return [qml.expval(qml.PauliZ(i)) for i in range(4)]

        @qml.qnode(scheduler_device)
        def scheduler_qnode(features):
            for i, value in enumerate(features):
                qml.RY(float(value) * np.pi, wires=i)
            qml.CNOT(wires=[0, 1]); qml.CNOT(wires=[1, 2]); qml.CNOT(wires=[2, 3])
            qml.RX(float(np.max(features)) * np.pi / 2.0, wires=0)
            qml.RZ(float(np.min(features)) * np.pi / 2.0, wires=3)
            return [qml.expval(qml.PauliZ(i)) for i in range(4)]

        @qml.qnode(memory_device)
        def memory_qnode(features):
            for i, value in enumerate(features):
                qml.RY(float(value) * np.pi, wires=i)
            qml.CNOT(wires=[0, 3]); qml.CNOT(wires=[1, 2])
            qml.RZ(float(np.mean(features)) * np.pi / 2.0, wires=2)
            return [qml.expval(qml.PauliZ(i)) for i in range(4)]

        @qml.qnode(reset_device)
        def reset_qnode(features):
            for i, value in enumerate(features):
                qml.RY(float(value) * np.pi, wires=i)
                qml.RZ(float(value) * np.pi / 2.0, wires=i)
            qml.CNOT(wires=[0, 1]); qml.CNOT(wires=[1, 2]); qml.CNOT(wires=[2, 3])
            qml.RX(float(np.mean(features)) * np.pi, wires=1)
            return [qml.expval(qml.PauliZ(i)) for i in range(4)]

        self.intent_qnode = intent_qnode
        self.task_qnode = task_qnode
        self.scheduler_qnode = scheduler_qnode
        self.memory_qnode = memory_qnode
        self.reset_qnode = reset_qnode

    def _surrogate(self, features: List[float], bias: float = 0.0) -> List[float]:
        vals = []
        for idx, value in enumerate(features):
            phase = (idx + 1) * 0.61 + bias
            vals.append(math.cos((value * math.pi) + phase) * 0.78)
        return _normalize_obs(vals)

    def intent_metrics(self, urgency: float, value: float, certainty: float, coordination_pressure: float) -> Dict[str, Any]:
        feats = [clamp(urgency), clamp(value), clamp(certainty), clamp(coordination_pressure)]
        obs = _normalize_obs(self.intent_qnode(np.array(feats))) if qml is not None else self._surrogate(feats, 0.11)
        return {"features": feats, "observables": obs, "intent_score": round(float(statistics.mean(obs)), 6)}

    def task_pigment_metrics(self, palette: ColorVector) -> Dict[str, Any]:
        d = palette.to_dict()
        feats = [
            clamp(d["gold"] + d["cyan"]),
            clamp(d["emerald"] + d["white"]),
            clamp(d["amber"] + d["crimson"]),
            clamp(d["violet"] + d["cyan"] / 2.0),
        ]
        obs = _normalize_obs(self.task_qnode(np.array(feats))) if qml is not None else self._surrogate(feats, 0.23)
        nonlinear = clamp((obs[0] * 0.34) + (obs[1] * 0.28) + (1.0 - obs[2]) * 0.24 + (obs[3] * 0.14))
        return {"features": feats, "observables": obs, "task_score": round(float(nonlinear), 6)}

    def scheduler_metrics(self, maintenance: float, reasoning: float, anomaly: float, delivery: float) -> Dict[str, Any]:
        feats = [clamp(maintenance), clamp(reasoning), clamp(anomaly), clamp(delivery)]
        obs = _normalize_obs(self.scheduler_qnode(np.array(feats))) if qml is not None else self._surrogate(feats, 0.37)
        best_idx = int(np.argmax(obs))
        return {
            "features": feats,
            "observables": obs,
            "mode": SCHEDULER_MODE_ORDER[best_idx],
            "mode_score": round(float(max(obs)), 6),
        }

    def memory_metrics(self, recency: float, stability: float, novelty: float, repair_success: float) -> Dict[str, Any]:
        feats = [clamp(recency), clamp(stability), clamp(novelty), clamp(repair_success)]
        obs = _normalize_obs(self.memory_qnode(np.array(feats))) if qml is not None else self._surrogate(feats, 0.49)
        return {"features": feats, "observables": obs, "memory_score": round(float(statistics.mean(obs)), 6)}

    def reset_metrics(self, ambiguity: float, overload: float, conflict: float, drift: float) -> Dict[str, Any]:
        feats = [clamp(ambiguity), clamp(overload), clamp(conflict), clamp(drift)]
        obs = _normalize_obs(self.reset_qnode(np.array(feats))) if qml is not None else self._surrogate(feats, 0.63)
        reset_score = clamp((obs[2] * 0.38) + (obs[3] * 0.34) + (obs[0] * 0.18) + (obs[1] * 0.10))
        return {"features": feats, "observables": obs, "reset_score": round(float(reset_score), 6)}

@dataclass
class WeightProfile:
    intent: float = 0.24
    task_pigment: float = 0.30
    memory: float = 0.18
    scheduler: float = 0.18
    colorsync: float = 0.10
    reset_threshold: float = 0.63
    colorsync_threshold: float = 0.16
    stabilization_alpha: float = 0.14
    memory_blend_alpha: float = 0.16

class QuantumCircuitCalibrator:
    def calibrate(self, recent_traces: List[Dict[str, Any]]) -> WeightProfile:
        if not recent_traces:
            return WeightProfile()
        reset_events = sum(1 for t in recent_traces if t["event_type"] == "reset")
        repair_events = sum(1 for t in recent_traces if "repair" in t["event_type"])
        colorsync_events = sum(1 for t in recent_traces if t["event_type"] == "colorsync")
        scale = max(len(recent_traces), 1)
        return WeightProfile(
            intent=round(clamp(0.22 + repair_events / (scale * 5), 0.18, 0.34), 6),
            task_pigment=round(clamp(0.28 + colorsync_events / (scale * 4), 0.24, 0.38), 6),
            memory=round(clamp(0.17 + repair_events / (scale * 6), 0.14, 0.28), 6),
            scheduler=round(clamp(0.17 + reset_events / (scale * 5), 0.14, 0.26), 6),
            colorsync=round(clamp(0.10 + colorsync_events / (scale * 6), 0.08, 0.18), 6),
            reset_threshold=round(clamp(0.60 + reset_events / (scale * 3), 0.55, 0.78), 6),
            colorsync_threshold=round(clamp(0.14 + colorsync_events / (scale * 5), 0.12, 0.30), 6),
            stabilization_alpha=round(clamp(0.12 + reset_events / (scale * 8), 0.10, 0.24), 6),
            memory_blend_alpha=round(clamp(0.14 + repair_events / (scale * 8), 0.10, 0.26), 6),
        )

@dataclass
class StarlingTaskIdea:
    title: str
    lane: str
    description: str
    dependencies: List[str] = field(default_factory=list)
    resources: List[str] = field(default_factory=list)
    priority: float = 0.7
    uncertainty: float = 0.35
    color_hint: Dict[str, float] = field(default_factory=dict)
    rationale: str = ""
    rare_condition: bool = False
    learning_value: float = 0.5

ADVANCED_RUNTIME_TASK_SYSTEM_PROMPT = """
You are the task-generator and plan-election strategist for a notebook-first autonomous codebase runtime.
Return strict JSON only.

Mission:
Design a dependency-aware task pack for a self-evolving multi-agent system that uses
chromatic cognition, ColorSync learning, MAPF scheduling, memory traces, verification,
repair loops, and PennyLane-ready quantum scoring.

Ordered planning procedure:
1. interpret the project objective and repository shape
2. identify the lane structure and the minimal control-surface tasks
3. add MAPF / reservation / handoff tasks
4. add quantum circuit and chromatic palette tasks
5. add memory-fusion / replay / calibration tasks
6. add rare-condition probes and anomaly-detection tasks
7. add verification, contracts, and repair-readiness tasks
8. add operator visibility / artifact / observability tasks
9. add at least two tasks explicitly about agents learning from other agents via ColorSync palettes
10. rank tasks in dependency order so the runtime can execute them safely

Hard requirements:
- JSON only
- Use only these lanes: architecture, backend, frontend, infra, test, docs
- include at least 8 and at most 14 task ideas
- at least 2 tasks must be agent-learning / ColorSync tasks
- at least 1 task must be a rare-condition task
- at least 1 task must be a verification task
- at least 1 task must be a repair task
- every task must include:
  title, lane, description, dependencies, resources, priority, uncertainty,
  color_hint, rationale, rare_condition, learning_value
- color_hint keys may only be: gold, emerald, amber, violet, crimson, white, cyan
- tasks must be safe for local development only

Return schema:
{
  "mode_hint": "reasoning | anomaly_detection | maintenance | delivery",
  "task_ideas": [...],
  "election_criteria": {
    "goal_alignment": 0.0,
    "lane_coverage": 0.0,
    "learning_density": 0.0,
    "repair_readiness": 0.0,
    "rare_condition_readiness": 0.0
  }
}
""".strip()

def build_advanced_task_prompt(spec: str, state: PlannerState, memory_digest: str, concept_bank: Dict[str, Any], max_tasks: int = 12) -> str:
    current_tasks = [
        {
            "task_id": t.task_id,
            "title": t.title,
            "lane": t.lane,
            "priority": t.priority,
            "uncertainty": t.uncertainty,
            "resources": t.resources,
            "rare_condition": t.rare_condition,
        }
        for t in list(state.tasks.values())[:24]
    ]
    prompt = f"""
{ADVANCED_RUNTIME_TASK_SYSTEM_PROMPT}

PROJECT_SPEC:
{spec}

CONCEPT_BANK_HEADINGS:
{json.dumps(concept_bank.get("headings", [])[:20], indent=2)}

CONCEPT_BANK_KEYWORDS:
{json.dumps(concept_bank.get("keywords", [])[:28], indent=2)}

CONCEPT_BANK_SUMMARY:
{concept_bank.get("summary", "")}

CURRENT_TASKS:
{json.dumps(current_tasks, indent=2)}

RECENT_MEMORY:
{memory_digest or "No recent memory traces."}

TASK_BUDGET:
Return at most {max_tasks} tasks.
""".strip()
    return prompt

def heuristic_task_pack(state: PlannerState) -> Dict[str, Any]:
    ideas = [
        StarlingTaskIdea(
            title="Create ColorSync mentorship graph",
            lane="architecture",
            description="Define how agents replay useful palette deltas from other agents after rare-condition discoveries, including trust weighting, time decay, and anti-herding safeguards.",
            dependencies=["T001"],
            resources=["contracts/colorsync_mentorship_graph.json", "docs/colorsync_learning.md"],
            priority=0.88,
            uncertainty=0.24,
            color_hint={"gold": 0.18, "cyan": 0.20, "violet": 0.16, "white": 0.10},
            rationale="Cross-agent learning needs explicit rules, not just opportunistic broadcasts.",
            rare_condition=False,
            learning_value=0.92,
        ),
        StarlingTaskIdea(
            title="Build rare-condition palette replay probe",
            lane="backend",
            description="Implement replay logic that captures the palette state around anomaly tasks and proposes safe palette updates for other agents that face similar preconditions.",
            dependencies=["T002", "T004"],
            resources=["src/runtime/palette_replay.py", "tests/test_palette_replay.py"],
            priority=0.87,
            uncertainty=0.28,
            color_hint={"crimson": 0.18, "amber": 0.18, "white": 0.12, "violet": 0.10},
            rationale="Rare conditions should become reusable learning assets rather than isolated events.",
            rare_condition=True,
            learning_value=0.95,
        ),
        StarlingTaskIdea(
            title="Add scheduler conflict atlas",
            lane="backend",
            description="Materialize a conflict atlas that maps resources, time windows, and agent routes into a reusable reservation graph for election and routing.",
            dependencies=["T003"],
            resources=["src/runtime/conflict_atlas.py", "contracts/conflict_atlas_schema.json"],
            priority=0.84,
            uncertainty=0.23,
            color_hint={"emerald": 0.20, "amber": 0.16, "gold": 0.12},
            rationale="MAPF assignments improve when conflict patterns are stored explicitly.",
            rare_condition=False,
            learning_value=0.66,
        ),
        StarlingTaskIdea(
            title="Implement palette outlier tribunal",
            lane="test",
            description="Add a review stage that examines an outlier palette before global propagation, checking whether the signal came from a valid rare condition or from noisy drift.",
            dependencies=["T005"],
            resources=["src/runtime/outlier_tribunal.py", "tests/test_outlier_tribunal.py"],
            priority=0.82,
            uncertainty=0.26,
            color_hint={"amber": 0.18, "violet": 0.18, "crimson": 0.12, "white": 0.14},
            rationale="ColorSync requires an evidence filter to avoid bad contagion.",
            rare_condition=True,
            learning_value=0.84,
        ),
        StarlingTaskIdea(
            title="Add memory-calibrated circuit tuner",
            lane="backend",
            description="Blend repair success, resets, and colorsync outcomes into learned weights for the circuit pack and document the tuning contract.",
            dependencies=["T002", "T004"],
            resources=["src/runtime/circuit_tuner.py", "contracts/circuit_weight_profile.json"],
            priority=0.86,
            uncertainty=0.22,
            color_hint={"emerald": 0.18, "white": 0.16, "gold": 0.12},
            rationale="Weights should adapt to prior runs and not remain fixed.",
            rare_condition=False,
            learning_value=0.78,
        ),
        StarlingTaskIdea(
            title="Create agent handoff election ledger",
            lane="architecture",
            description="Score competing handoff paths using lane fit, palette compatibility, reservation pressure, and verification confidence.",
            dependencies=["T003", "T005"],
            resources=["contracts/handoff_election_ledger.json", "docs/handoff_election.md"],
            priority=0.83,
            uncertainty=0.24,
            color_hint={"gold": 0.18, "white": 0.16, "emerald": 0.12, "amber": 0.10},
            rationale="The system needs an explicit election mechanism for candidate execution routes.",
            rare_condition=False,
            learning_value=0.71,
        ),
        StarlingTaskIdea(
            title="Publish operator anomaly review board",
            lane="docs",
            description="Create an operator-facing explanation of anomaly triggers, reset events, color outliers, and why the elected plan won.",
            dependencies=["T006"],
            resources=["docs/anomaly_review_board.md", "reports/plan_election_report.md"],
            priority=0.79,
            uncertainty=0.19,
            color_hint={"white": 0.22, "gold": 0.16, "violet": 0.12},
            rationale="Human operators should be able to inspect the runtime's reasoning surfaces.",
            rare_condition=False,
            learning_value=0.52,
        ),
    ]
    return {
        "mode_hint": "reasoning",
        "task_ideas": [asdict(x) for x in ideas],
        "election_criteria": {
            "goal_alignment": 0.91,
            "lane_coverage": 0.82,
            "learning_density": 0.93,
            "repair_readiness": 0.88,
            "rare_condition_readiness": 0.86,
        },
    }

def parse_task_pack(payload: Dict[str, Any], start_index: int = 100) -> Tuple[str, List[TaskNode], Dict[str, Any]]:
    mode_hint = payload.get("mode_hint", "reasoning")
    ideas = payload.get("task_ideas", []) or []
    out = []
    title_to_task_id = {}
    cursor = start_index
    for item in ideas:
        cursor += 1
        task_id = f"T{cursor:03d}"
        title = str(item.get("title", "")).strip()
        lane = str(item.get("lane", "backend")).strip()
        if not title or lane not in LANE_ORDER:
            continue
        task = TaskNode(
            task_id=task_id,
            title=title,
            lane=lane,
            description=str(item.get("description", "")).strip(),
            dependencies=[],
            resources=[str(x) for x in item.get("resources", [])],
            priority=clamp(item.get("priority", 0.72)),
            uncertainty=clamp(item.get("uncertainty", 0.35)),
            source="generated",
            color_hint=normalize_dict(item.get("color_hint", {})),
            rationale=str(item.get("rationale", "")).strip(),
            rare_condition=bool(item.get("rare_condition", False)),
        )
        out.append(task)
        title_to_task_id[title] = task_id

    # second pass for dependency title resolution
    for task, item in zip(out, ideas):
        deps = []
        for dep in item.get("dependencies", []):
            dep = str(dep)
            if dep in title_to_task_id:
                deps.append(title_to_task_id[dep])
            elif dep in {"T001", "T002", "T003", "T004", "T005", "T006"}:
                deps.append(dep)
        task.dependencies = deps

    election = payload.get("election_criteria", {})
    return mode_hint, out, election

def maybe_generate_starling_task_pack(prompt: str) -> Tuple[str, Dict[str, Any]]:
    text = llm_complete(prompt)
    payload = extract_json_block(text)
    if payload:
        return "starling", payload
    return "heuristic", {}

qpack = QuantumCircuitPack()
calibrator = QuantumCircuitCalibrator()

# Preview the advanced prompt
_preview_state = PlannerState(spec=DEFAULT_SPEC)
_preview_state.add_tasks(deterministic_seed_tasks(DEFAULT_SPEC))
ADVANCED_PROMPT_PREVIEW = build_advanced_task_prompt(
    spec=DEFAULT_SPEC,
    state=_preview_state,
    memory_digest="",
    concept_bank=concept_bank,
    max_tasks=12,
)
print(ADVANCED_PROMPT_PREVIEW[:2200] + "\n\n... [truncated preview] ...")



# ---- Ordered concept script for Starling / local LLM task generation ----
CONCEPT_SCRIPT_ITEMS = [
    {
        "name": "scheduling",
        "purpose": "Use dependency-safe MAPF coordination with reservation ledgers, time windows, handoffs, and conflict-aware routing.",
        "operator_question": "Which resources, paths, or files can collide, and how do we reserve them safely across steps?",
        "must_show_up_as_tasks": [
            "lane structure and control surface",
            "MAPF reservation ledger",
            "handoff election",
            "conflict atlas or routing graph",
        ],
    },
    {
        "name": "memory traces",
        "purpose": "Record task execution, palette broadcasts, resets, failures, repair results, and election outcomes in replayable traces.",
        "operator_question": "What should the runtime remember so later runs can calibrate plans instead of starting cold?",
        "must_show_up_as_tasks": [
            "trace schema",
            "memory fusion or replay",
            "calibration from prior runs",
            "trace digest for prompts",
        ],
    },
    {
        "name": "verification",
        "purpose": "Add explicit verification gates for syntax, contracts, workspace integrity, test readiness, and observability outputs.",
        "operator_question": "How will the runtime know a generated task pack or workspace is actually coherent?",
        "must_show_up_as_tasks": [
            "verification harness",
            "contract checks",
            "test readiness",
            "artifact validation",
        ],
    },
    {
        "name": "repair_loops",
        "purpose": "Treat repairs as first-class planner outputs with recovery tasks, patch proposals, re-verification, and safe rollback thinking.",
        "operator_question": "If verification fails, what is the next safe repair action and how do we prove it improved the state?",
        "must_show_up_as_tasks": [
            "repair synthesis",
            "patch generation or proposal",
            "post-repair verification",
            "repair-readiness scoring",
        ],
    },
    {
        "name": "pennylane_ready_quantum_scoring",
        "purpose": "Use chromatic palette state plus PennyLane-ready intent, task pigment, scheduler, memory, and reset circuits for nonlinear scoring.",
        "operator_question": "How should the runtime combine urgency, ambiguity, anomaly pressure, memory, and resolution into one score surface?",
        "must_show_up_as_tasks": [
            "intent circuit",
            "task pigment circuit",
            "scheduler circuit",
            "memory circuit",
            "reset circuit",
            "chromatic palette tasks",
        ],
    },
    {
        "name": "colorsync_learning",
        "purpose": "Ensure agents learn from other agents through palette broadcasts, outlier review, replay, trust weighting, and rare-condition propagation.",
        "operator_question": "How will one agent's rare discovery become safe shared learning for the rest of the system?",
        "must_show_up_as_tasks": [
            "at least two ColorSync learning tasks",
            "outlier tribunal",
            "palette replay",
            "trust-weighted propagation",
        ],
    },
]

ORDERED_PLANNING_PROCEDURE = [
    "interpret the project objective and repository shape",
    "identify the lane structure and the minimal control-surface tasks",
    "add MAPF / reservation / handoff tasks",
    "add quantum circuit and chromatic palette tasks",
    "add memory-fusion / replay / calibration tasks",
    "add rare-condition probes and anomaly-detection tasks",
    "add verification, contracts, and repair-readiness tasks",
    "add operator visibility / artifact / observability tasks",
    "add at least two tasks explicitly about agents learning from other agents via ColorSync palettes",
    "rank tasks in dependency order so the runtime can execute them safely",
]

CONCEPT_SCRIPT_TEXT = "\n\n".join(
    [
        "CONCEPT: {name}\nPURPOSE: {purpose}\nOPERATOR_QUESTION: {operator_question}\nTASK_IMPLICATIONS:\n- {tasks}".format(
            name=item["name"],
            purpose=item["purpose"],
            operator_question=item["operator_question"],
            tasks="\n- ".join(item["must_show_up_as_tasks"]),
        )
        for item in CONCEPT_SCRIPT_ITEMS
    ]
)

ADVANCED_RUNTIME_TASK_SYSTEM_PROMPT = textwrap.dedent(
    """
    You are the task-generator and plan-election strategist for a notebook-first autonomous codebase runtime.
    Return strict JSON only.

    Mission:
    Design a dependency-aware task pack for a self-evolving multi-agent system that uses
    chromatic cognition, ColorSync learning, MAPF scheduling, memory traces, verification,
    repair loops, and PennyLane-ready quantum scoring.

    Concept script (apply each concept one by one):
    {concept_script}

    Ordered planning procedure:
    1. {step1}
    2. {step2}
    3. {step3}
    4. {step4}
    5. {step5}
    6. {step6}
    7. {step7}
    8. {step8}
    9. {step9}
    10. {step10}

    Hard requirements:
    - JSON only
    - Use only these lanes: architecture, backend, frontend, infra, test, docs
    - include at least 10 and at most 16 task ideas
    - at least 2 tasks must be agent-learning / ColorSync tasks
    - at least 1 task must be a rare-condition task
    - at least 1 task must be a verification task
    - at least 1 task must be a repair task
    - at least 1 task must explicitly improve scheduling / MAPF coordination
    - at least 1 task must explicitly improve memory traces or replay
    - at least 1 task must explicitly improve PennyLane-ready quantum scoring
    - every task must include:
      title, lane, description, dependencies, resources, priority, uncertainty,
      color_hint, rationale, rare_condition, learning_value
    - color_hint keys may only be: gold, emerald, amber, violet, crimson, white, cyan
    - tasks must be safe for local development only
    - dependencies should prefer earlier control-surface tasks before deeper implementation tasks
    - do not collapse multiple concepts into a single vague task; expose visible control-surface work

    Return schema:
    {{
      "mode_hint": "reasoning | anomaly_detection | maintenance | delivery",
      "task_ideas": [...],
      "election_criteria": {{
        "goal_alignment": 0.0,
        "lane_coverage": 0.0,
        "learning_density": 0.0,
        "repair_readiness": 0.0,
        "rare_condition_readiness": 0.0
      }}
    }}
    """
).format(
    concept_script=CONCEPT_SCRIPT_TEXT,
    step1=ORDERED_PLANNING_PROCEDURE[0],
    step2=ORDERED_PLANNING_PROCEDURE[1],
    step3=ORDERED_PLANNING_PROCEDURE[2],
    step4=ORDERED_PLANNING_PROCEDURE[3],
    step5=ORDERED_PLANNING_PROCEDURE[4],
    step6=ORDERED_PLANNING_PROCEDURE[5],
    step7=ORDERED_PLANNING_PROCEDURE[6],
    step8=ORDERED_PLANNING_PROCEDURE[7],
    step9=ORDERED_PLANNING_PROCEDURE[8],
    step10=ORDERED_PLANNING_PROCEDURE[9],
).strip()


def build_advanced_task_prompt(spec: str, state: PlannerState, memory_digest: str, concept_bank: Dict[str, Any], max_tasks: int = 14) -> str:
    current_tasks = [
        {
            "task_id": t.task_id,
            "title": t.title,
            "lane": t.lane,
            "priority": t.priority,
            "uncertainty": t.uncertainty,
            "resources": t.resources,
            "rare_condition": t.rare_condition,
            "dependencies": t.dependencies,
            "rationale": t.rationale,
        }
        for t in list(state.tasks.values())[:28]
    ]
    prompt = f"""
{ADVANCED_RUNTIME_TASK_SYSTEM_PROMPT}

PROJECT_SPEC:
{spec}

CONCEPT_BANK_HEADINGS:
{json.dumps(concept_bank.get("headings", [])[:20], indent=2)}

CONCEPT_BANK_KEYWORDS:
{json.dumps(concept_bank.get("keywords", [])[:32], indent=2)}

CONCEPT_BANK_SUMMARY:
{concept_bank.get("summary", "")}

CURRENT_TASKS:
{json.dumps(current_tasks, indent=2)}

RECENT_MEMORY:
{memory_digest or "No recent memory traces."}

ORDERED_CONCEPT_SCRIPT_JSON:
{json.dumps(CONCEPT_SCRIPT_ITEMS, indent=2)}

ORDERED_PLANNING_PROCEDURE_JSON:
{json.dumps(ORDERED_PLANNING_PROCEDURE, indent=2)}

TASK_BUDGET:
Return at most {max_tasks} tasks.
""".strip()
    return prompt


def heuristic_task_pack(state: PlannerState) -> Dict[str, Any]:
    ideas = [
        StarlingTaskIdea(
            title="Map repository shape and runtime control surface",
            lane="architecture",
            description="Interpret the project objective, define repository zones, and enumerate the minimal control-surface tasks that the runtime must own before deeper generation begins.",
            dependencies=["T001"],
            resources=["docs/repository_shape.md", "contracts/control_surface.json"],
            priority=0.93,
            uncertainty=0.18,
            color_hint={"gold": 0.18, "white": 0.18, "emerald": 0.12, "cyan": 0.10},
            rationale="The ordered planning procedure starts by clarifying repository shape and control surfaces.",
            rare_condition=False,
            learning_value=0.68,
        ),
        StarlingTaskIdea(
            title="Create lane topology and handoff election map",
            lane="architecture",
            description="Define lane boundaries, ownership, and the safe handoff graph so the runtime can reason about where work should flow next.",
            dependencies=["Map repository shape and runtime control surface"],
            resources=["contracts/lane_topology.json", "docs/handoff_topology.md"],
            priority=0.91,
            uncertainty=0.20,
            color_hint={"gold": 0.16, "white": 0.16, "emerald": 0.14, "amber": 0.10},
            rationale="The runtime should expose lane structure before execution and plan election.",
            rare_condition=False,
            learning_value=0.71,
        ),
        StarlingTaskIdea(
            title="Build MAPF reservation and scheduling atlas",
            lane="backend",
            description="Implement reservation-aware scheduling, conflict atlas generation, file/resource constraints, and time-window routing for concurrent agents.",
            dependencies=["Create lane topology and handoff election map"],
            resources=["src/runtime/conflict_atlas.py", "contracts/reservation_atlas.json"],
            priority=0.90,
            uncertainty=0.22,
            color_hint={"emerald": 0.18, "amber": 0.18, "gold": 0.12, "white": 0.10},
            rationale="Scheduling must be explicit and conflict-aware, not implicit.",
            rare_condition=False,
            learning_value=0.74,
        ),
        StarlingTaskIdea(
            title="Implement chromatic palette schema and task pigment routing",
            lane="backend",
            description="Define chromatic palette contracts and route task evaluation through palette-aware scoring surfaces so tasks can carry mixed cognitive signals.",
            dependencies=["Map repository shape and runtime control surface"],
            resources=["contracts/chromatic_palette_schema.json", "src/runtime/task_pigment.py"],
            priority=0.89,
            uncertainty=0.24,
            color_hint={"violet": 0.16, "cyan": 0.18, "gold": 0.14, "emerald": 0.12},
            rationale="Chromatic routing is a first-class part of the planner, not just a visualization layer.",
            rare_condition=False,
            learning_value=0.76,
        ),
        StarlingTaskIdea(
            title="Implement PennyLane-ready quantum circuit bundle",
            lane="backend",
            description="Provide intent, task pigment, scheduler, memory, and reset circuits with a stable API that can run on a surrogate backend now and PennyLane later.",
            dependencies=["Implement chromatic palette schema and task pigment routing"],
            resources=["src/runtime/quantum_bundle.py", "tests/test_quantum_bundle.py"],
            priority=0.92,
            uncertainty=0.23,
            color_hint={"gold": 0.16, "violet": 0.16, "cyan": 0.14, "emerald": 0.12},
            rationale="Quantum scoring should remain visible as its own implementation target.",
            rare_condition=False,
            learning_value=0.82,
        ),
        StarlingTaskIdea(
            title="Add memory-trace schema and replay digest",
            lane="backend",
            description="Store execution traces, palette broadcasts, resets, repairs, and election outcomes in a replayable schema with prompt-ready digests.",
            dependencies=["Map repository shape and runtime control surface"],
            resources=["src/runtime/memory_trace.py", "contracts/memory_trace_schema.json"],
            priority=0.91,
            uncertainty=0.20,
            color_hint={"emerald": 0.18, "white": 0.18, "violet": 0.10, "amber": 0.08},
            rationale="Memory traces are required for calibration and for future runs to learn from current runs.",
            rare_condition=False,
            learning_value=0.81,
        ),
        StarlingTaskIdea(
            title="Create memory-fusion calibration tuner",
            lane="backend",
            description="Blend replay statistics, repair outcomes, resets, and colorsync signals into learned weights and thresholds for the circuit pack.",
            dependencies=["Add memory-trace schema and replay digest", "Implement PennyLane-ready quantum circuit bundle"],
            resources=["src/runtime/calibration_tuner.py", "contracts/circuit_calibration_profile.json"],
            priority=0.88,
            uncertainty=0.22,
            color_hint={"emerald": 0.16, "white": 0.16, "gold": 0.12, "amber": 0.08},
            rationale="Replay should directly improve later scoring and stabilization.",
            rare_condition=False,
            learning_value=0.84,
        ),
        StarlingTaskIdea(
            title="Add rare-condition anomaly probes",
            lane="test",
            description="Probe for unusual task preconditions, outlier palettes, and scheduler conflicts that may signal rare conditions requiring coordinated response.",
            dependencies=["Build MAPF reservation and scheduling atlas", "Implement PennyLane-ready quantum circuit bundle"],
            resources=["src/runtime/anomaly_probes.py", "tests/test_anomaly_probes.py"],
            priority=0.87,
            uncertainty=0.26,
            color_hint={"crimson": 0.18, "amber": 0.18, "violet": 0.14, "white": 0.10},
            rationale="Rare-condition detection should precede wide propagation or irreversible execution.",
            rare_condition=True,
            learning_value=0.86,
        ),
        StarlingTaskIdea(
            title="Create ColorSync mentorship graph",
            lane="architecture",
            description="Define how agents learn from other agents by viewing ColorSync palette deltas, applying trust weighting, time decay, and anti-herding safeguards.",
            dependencies=["Create lane topology and handoff election map", "Add memory-trace schema and replay digest"],
            resources=["contracts/colorsync_mentorship_graph.json", "docs/colorsync_learning.md"],
            priority=0.89,
            uncertainty=0.24,
            color_hint={"gold": 0.16, "cyan": 0.20, "violet": 0.16, "white": 0.10},
            rationale="The ordered procedure explicitly requires at least two ColorSync learning tasks.",
            rare_condition=False,
            learning_value=0.93,
        ),
        StarlingTaskIdea(
            title="Build rare-condition palette replay probe",
            lane="backend",
            description="Capture palette state around anomalies and propose safe shared updates when other agents encounter similar preconditions in later steps.",
            dependencies=["Create ColorSync mentorship graph", "Add rare-condition anomaly probes"],
            resources=["src/runtime/palette_replay.py", "tests/test_palette_replay.py"],
            priority=0.90,
            uncertainty=0.27,
            color_hint={"crimson": 0.18, "amber": 0.18, "white": 0.12, "violet": 0.10},
            rationale="Rare discoveries should become reusable learning assets across agents.",
            rare_condition=True,
            learning_value=0.95,
        ),
        StarlingTaskIdea(
            title="Implement outlier tribunal for palette propagation",
            lane="test",
            description="Review palette outliers before global propagation and decide whether they represent meaningful anomalies or noisy drift.",
            dependencies=["Build rare-condition palette replay probe"],
            resources=["src/runtime/outlier_tribunal.py", "tests/test_outlier_tribunal.py"],
            priority=0.85,
            uncertainty=0.25,
            color_hint={"amber": 0.18, "violet": 0.18, "crimson": 0.10, "white": 0.12},
            rationale="ColorSync should be filtered through evidence, not blindly copied.",
            rare_condition=True,
            learning_value=0.88,
        ),
        StarlingTaskIdea(
            title="Create verification and contract gate suite",
            lane="test",
            description="Run syntax checks, contract checks, observability checks, and test-readiness gates over the generated workspace and exported artifacts.",
            dependencies=["Implement PennyLane-ready quantum circuit bundle", "Build MAPF reservation and scheduling atlas", "Add memory-trace schema and replay digest"],
            resources=["src/runtime/verify.py", "tests/test_runtime_verify.py"],
            priority=0.91,
            uncertainty=0.23,
            color_hint={"amber": 0.18, "white": 0.18, "emerald": 0.12, "crimson": 0.10},
            rationale="Verification must be visible, explicit, and broad enough to catch drift.",
            rare_condition=False,
            learning_value=0.80,
        ),
        StarlingTaskIdea(
            title="Add repair synthesis and re-verification loop",
            lane="backend",
            description="Synthesize recovery tasks, patch proposals, and post-repair verification runs so failures automatically feed a controlled repair loop.",
            dependencies=["Create verification and contract gate suite"],
            resources=["src/runtime/repair_loop.py", "tests/test_repair_loop.py"],
            priority=0.90,
            uncertainty=0.24,
            color_hint={"white": 0.20, "amber": 0.18, "crimson": 0.12, "emerald": 0.10},
            rationale="Repair loops should be first-class planner outputs rather than ad hoc followups.",
            rare_condition=False,
            learning_value=0.85,
        ),
        StarlingTaskIdea(
            title="Publish operator observability and election board",
            lane="docs",
            description="Materialize operator-facing artifacts that explain the winning plan, trace summaries, anomalies, resets, verification outcomes, and repair readiness.",
            dependencies=["Create verification and contract gate suite", "Add repair synthesis and re-verification loop"],
            resources=["docs/operator_board.md", "reports/plan_election_report.md"],
            priority=0.82,
            uncertainty=0.18,
            color_hint={"white": 0.22, "gold": 0.16, "violet": 0.10},
            rationale="The runtime should remain inspectable to operators at every stage.",
            rare_condition=False,
            learning_value=0.58,
        ),
    ]
    return {
        "mode_hint": "reasoning",
        "task_ideas": [asdict(x) for x in ideas],
        "election_criteria": {
            "goal_alignment": 0.94,
            "lane_coverage": 0.89,
            "learning_density": 0.96,
            "repair_readiness": 0.91,
            "rare_condition_readiness": 0.90,
        },
    }

# refresh preview after overrides
_preview_state = PlannerState(spec=DEFAULT_SPEC)
_preview_state.add_tasks(deterministic_seed_tasks(DEFAULT_SPEC))
ADVANCED_PROMPT_PREVIEW = build_advanced_task_prompt(
    spec=DEFAULT_SPEC,
    state=_preview_state,
    memory_digest="",
    concept_bank=concept_bank,
    max_tasks=14,
)
print(CONCEPT_SCRIPT_TEXT[:1800] + "\n\n---\n")
print(ADVANCED_PROMPT_PREVIEW[:2400] + "\n\n... [truncated preview] ...")


In [ ]:

# @title 6) Prompt grammar upgrade + long-memory advancement layer
from collections import Counter, defaultdict

PROMPT_TAG_GUIDE = {
    "response": "outer response envelope",
    "json": "strict machine-readable body",
    "action": "ordered planning action directives",
    "memory_packet": "long-horizon memory briefing",
    "concept": "one concept at a time, with operational meaning",
    "constraint": "hard safety and formatting rules",
    "rarity_probe": "forces at least one rare-condition task",
    "mentor_exchange": "forces agent-learning / ColorSync tasks",
    "verification_gate": "forces verification and repair readiness",
    "operator_visibility": "forces observability artifacts",
}

PROMPT_INVENTIONS = [
    "palette rumor quarantine to prevent bad ColorSync contagion",
    "mentor quorum replay so multiple agents can validate a rare palette before adoption",
    "memory braid that fuses resets, repairs, colorsync events, and election outcomes",
    "counterfactual archive that stores near-miss plans for later calibration",
    "stability ladder that requires verify -> repair -> reverify ordering",
    "anomaly witness packet that captures what changed before a rare condition surfaced",
    "role-locked execution grammar so the planner does not drift into prose",
]

MEMORY_ADVANCEMENT_ITEMS = [
    {
        "name": "episodic_memory_braid",
        "purpose": "Fuse resets, failures, recoveries, colorsync broadcasts, and election wins into a replayable memory thread.",
        "task_implications": [
            "episode braid schema",
            "trace stitching",
            "cross-run replay packet",
            "memory confidence tags",
        ],
    },
    {
        "name": "rare_condition_capsules",
        "purpose": "Save anomaly preconditions, palette deltas, and post-repair outcomes as reusable rare-condition capsules.",
        "task_implications": [
            "rare precondition capture",
            "palette delta archive",
            "repair outcome capsule",
            "safe propagation thresholds",
        ],
    },
    {
        "name": "mentor_quorum_memory",
        "purpose": "Record which agents taught which others, under what trust levels, and whether the teaching improved future execution.",
        "task_implications": [
            "mentor lattice",
            "teaching efficacy ledger",
            "trust decay or reinforcement",
            "palette mentorship replay",
        ],
    },
    {
        "name": "counterfactual_archive",
        "purpose": "Store rejected plan branches and failed patch ideas so the planner can calibrate future elections.",
        "task_implications": [
            "counterfactual archive",
            "rejected plan summary",
            "near-miss analysis",
            "plan election memory hooks",
        ],
    },
]

# Extend the concept script one-by-one with longer memory emphasis and prompt grammar
CONCEPT_SCRIPT_ITEMS = CONCEPT_SCRIPT_ITEMS + [
    {
        "name": "long_horizon_memory_advancement",
        "purpose": "Upgrade memory from a short trace log into an episodic braid with rare-condition capsules, mentor quorum memory, and counterfactual archives.",
        "operator_question": "What should the runtime preserve so future elections become wiser instead of merely larger?",
        "must_show_up_as_tasks": [
            "episodic memory braid",
            "rare-condition capsule",
            "mentor quorum memory",
            "counterfactual archive",
        ],
    },
    {
        "name": "prompt_grammar_and_execution_tokens",
        "purpose": "Use [action] and companion control tags to force coherent, ordered, machine-parseable planner output.",
        "operator_question": "How do we keep the local LLM responsive and on-format without letting it drift into generic prose?",
        "must_show_up_as_tasks": [
            "tagged prompt envelope",
            "strict JSON response body",
            "ordered action blocks",
            "response validation tokens",
        ],
    },
]

CONCEPT_SCRIPT_TEXT = "\n\n".join(
    [
        "[concept index=\"{idx}\" name=\"{name}\"]\nPURPOSE: {purpose}\nOPERATOR_QUESTION: {operator_question}\nTASK_IMPLICATIONS:\n- {tasks}\n[/concept]".format(
            idx=i + 1,
            name=item["name"],
            purpose=item["purpose"],
            operator_question=item["operator_question"],
            tasks="\n- ".join(item["must_show_up_as_tasks"]),
        )
        for i, item in enumerate(CONCEPT_SCRIPT_ITEMS)
    ]
)

MEMORY_ADVANCEMENT_GUIDE_TEXT = "\n\n".join(
    [
        "[memory_advancement name=\"{name}\"]\nPURPOSE: {purpose}\nTASK_IMPLICATIONS:\n- {tasks}\n[/memory_advancement]".format(
            name=item["name"],
            purpose=item["purpose"],
            tasks="\n- ".join(item["task_implications"]),
        )
        for item in MEMORY_ADVANCEMENT_ITEMS
    ]
)

def _tag(tag: str, content: str = "", **attrs: Any) -> str:
    attr_text = " ".join(f'{k}="{v}"' for k, v in attrs.items() if v is not None)
    prefix = f"[{tag}{(' ' + attr_text) if attr_text else ''}]"
    suffix = f"[/{tag}]"
    body = (content or "").strip()
    return f"{prefix}\n{body}\n{suffix}"

def build_memory_prompt_packet(memory: MemoryStore, limit: int = 80, concept_script_items: Optional[List[Dict[str, Any]]] = None) -> str:
    traces = memory.recent_traces(limit=limit)
    counts = Counter(item.get("event_type", "") for item in traces)
    phase_counts = Counter(item.get("phase", "") for item in traces)
    lane_counts = Counter(item.get("lane", "") for item in traces if item.get("lane"))
    recent_lines = []
    rare_lines = []
    mentor_lines = []
    repair_lines = []
    counterfactual_lines = []
    for item in traces[: min(len(traces), 24)]:
        payload = item.get("payload", {}) or {}
        note = payload.get("note") or payload.get("detail") or payload.get("summary") or json.dumps(payload, sort_keys=True)[:220]
        line = f"- phase={item.get('phase')} event={item.get('event_type')} task={item.get('task_id') or '-'} lane={item.get('lane') or '-'} note={note}"
        recent_lines.append(line)
        if item.get("event_type") in {"colorsync", "task_failed", "reset"} or payload.get("rare_condition"):
            rare_lines.append(line)
        if item.get("event_type") in {"colorsync", "teaching", "mentor"} or "trust" in json.dumps(payload):
            mentor_lines.append(line)
        if item.get("event_type") in {"task_failed", "repair", "repair_generated", "verification"} or "repair" in note.lower():
            repair_lines.append(line)
        if item.get("event_type") in {"plan_rejected", "counterfactual", "task_failed"}:
            counterfactual_lines.append(line)

    lessons = []
    if counts.get("task_failed", 0) > 0:
        lessons.append("Verification and repair-readiness should move earlier in the dependency chain.")
    if counts.get("reset", 0) > 0:
        lessons.append("The runtime has experienced instability; add stabilization, anomaly probes, and reset-aware memory tasks.")
    if counts.get("colorsync", 0) > 0:
        lessons.append("Cross-agent learning is active; include trust-weighted palette replay and outlier review tasks.")
    if counts.get("task_completed", 0) >= counts.get("task_failed", 0):
        lessons.append("Stable execution exists; keep Emerald/White-weighted verification and artifact delivery tasks.")
    if not lessons:
        lessons.append("Cold start memory: seed memory braid, replay, and calibration tasks so later runs have more signal.")

    concept_lines = []
    for item in (concept_script_items or [])[:10]:
        concept_lines.append(f"- {item['name']}: {item['purpose']}")

    packet_parts = [
        _tag("memory_packet",
             "\n".join([
                 "EVENT_COUNTS: " + json.dumps(dict(counts), sort_keys=True),
                 "PHASE_COUNTS: " + json.dumps(dict(phase_counts), sort_keys=True),
                 "LANE_COUNTS: " + json.dumps(dict(lane_counts), sort_keys=True),
             ]),
             kind="histogram"),
        _tag("memory_lessons", "\n".join(f"- {x}" for x in lessons)),
        _tag("memory_recent", "\n".join(recent_lines or ["- no recent traces available"])),
        _tag("rare_condition_memory", "\n".join(rare_lines[:12] or ["- no rare-condition traces yet"])),
        _tag("mentor_memory", "\n".join(mentor_lines[:12] or ["- no mentor traces yet"])),
        _tag("repair_memory", "\n".join(repair_lines[:12] or ["- no repair traces yet"])),
        _tag("counterfactual_memory", "\n".join(counterfactual_lines[:12] or ["- no counterfactual traces yet"])),
        _tag("memory_advancement_guide", MEMORY_ADVANCEMENT_GUIDE_TEXT),
        _tag("concept_memory_alignment", "\n".join(concept_lines or ["- no concept script items loaded"])),
    ]
    return "\n\n".join(packet_parts)

def extract_json_block(text: str) -> Dict[str, Any]:
    text = (text or "").strip()
    if not text:
        return {}
    patterns = [
        r"\[json\](.*?)(?:\[/json\]|$)",
        r"\[response\].*?\[json\](.*?)(?:\[/json\]|$).*?(?:\[/response\]|$)",
        r"```json\s*(.*?)```",
        r"(\{.*\})",
    ]
    for pattern in patterns:
        for candidate in re.findall(pattern, text, flags=re.S | re.I):
            candidate = candidate.strip()
            if candidate.startswith("[json]"):
                candidate = re.sub(r"^\[json\]\s*", "", candidate)
            try:
                return json.loads(candidate)
            except Exception:
                continue
    return {}

def llm_complete(prompt: str) -> str:
    model = load_llm()
    if model is None:
        return ""
    out = model(
        trim_prompt(prompt, limit_chars=18000),
        max_tokens=int(STARLING_MAX_TOKENS),
        temperature=float(STARLING_TEMPERATURE),
        top_p=0.90,
        repeat_penalty=1.10,
        stop=["[/response]", "</end>", "\n\n\n\n"],
    )
    text = out["choices"][0]["text"].strip()
    if text and text.lstrip().startswith("{"):
        text = "[response]\n[json]\n" + text + "\n[/json]\n[/response]"
    return text

ADVANCED_RUNTIME_TASK_SYSTEM_PROMPT = _tag(
    "system_role",
    "You are Starling acting as the task-generator, plan-election strategist, and runtime-shaping architect for a notebook-first autonomous codebase runtime."
) + "\n\n" + _tag(
    "mission",
    "Produce a dependency-safe task pack for a self-evolving multi-agent system that combines chromatic cognition, ColorSync learning, MAPF scheduling, memory traces, verification, repair loops, and PennyLane-ready quantum scoring."
) + "\n\n" + _tag(
    "prompt_inventions",
    "\n".join(f"- {item}" for item in PROMPT_INVENTIONS)
) + "\n\n" + _tag(
    "constraint",
    "\n".join([
        "Return machine-parseable JSON only inside [response] and [json] tags.",
        "Use only these lanes: architecture, backend, frontend, infra, test, docs.",
        "Include at least 10 and at most 16 task ideas.",
        "At least 2 tasks must be explicit ColorSync / agent-learning tasks.",
        "At least 2 tasks must be long-memory-advancement tasks.",
        "At least 1 task must be a rare-condition task.",
        "At least 1 task must be a verification task.",
        "At least 1 task must be a repair task.",
        "Every task must include title, lane, description, dependencies, resources, priority, uncertainty, color_hint, rationale, rare_condition, learning_value.",
        "color_hint keys may only be: gold, emerald, amber, violet, crimson, white, cyan.",
        "Tasks must remain safe for local development only.",
    ]),
    level="hard"
)

def build_ordered_action_script(max_tasks: int = 14) -> str:
    action_map = [
        ("interpret_objective", "Read the project objective, infer repository shape, and name the minimal runtime control surface."),
        ("elect_lanes", "Pick the lane structure and the smallest dependency-safe set of control-surface tasks."),
        ("mapf_and_handoffs", "Add MAPF reservation, routing, and handoff election tasks before broad feature work."),
        ("quantum_and_chromatic", "Add intent, task pigment, scheduler, memory, and reset circuit tasks, plus chromatic palette tasks."),
        ("memory_fusion", "Add memory braid, replay, calibration, mentor memory, and rare-condition capsule tasks."),
        ("rare_probes", "Inject rare-condition probes and anomaly witness tasks that capture preconditions and propagation controls."),
        ("verification_gate", "Insert syntax, contracts, test readiness, artifact validation, and observability verification tasks."),
        ("repair_readiness", "Insert repair synthesis, patch proposal, rollback thinking, and re-verification tasks."),
        ("operator_visibility", "Add operator dashboards, explanation graphs, election reports, and runtime artifacts."),
        ("colorsync_learning", "Create at least two explicit tasks about agents learning from other agents via ColorSync palettes."),
        ("dependency_rank", "Rank all tasks in dependency-safe order for execution."),
    ]
    blocks = []
    for i, (name, instruction) in enumerate(action_map, start=1):
        blocks.append(_tag("action", instruction, id=f"{i:02d}", name=name))
    blocks.append(_tag("rarity_probe", "Invent at least one task that only becomes obviously necessary after a rare condition or anomaly signal appears."))
    blocks.append(_tag("mentor_exchange", "Invent at least two tasks where one agent teaches other agents through palette replay, trust weighting, or outlier review."))
    blocks.append(_tag("verification_gate", "Ensure the final dependency order forces verification before closure and repair before declaring recovery."))
    blocks.append(_tag("operator_visibility", "Ensure at least one docs or operator task makes the election and runtime observable to humans."))
    blocks.append(_tag("task_budget", f"Return at most {max_tasks} tasks, but prefer 12-15 when the repository shape supports it."))
    return "\n\n".join(blocks)

def build_advanced_task_prompt(spec: str, state: PlannerState, memory_digest: str, concept_bank: Dict[str, Any], max_tasks: int = 14) -> str:
    current_tasks = [
        {
            "task_id": t.task_id,
            "title": t.title,
            "lane": t.lane,
            "description": t.description[:180],
            "priority": t.priority,
            "uncertainty": t.uncertainty,
            "resources": t.resources,
            "dependencies": t.dependencies,
            "rare_condition": t.rare_condition,
            "color_hint": t.color_hint,
        }
        for t in list(state.tasks.values())[:32]
    ]
    concept_block = _tag("concept_script", CONCEPT_SCRIPT_TEXT)
    concept_headings = _tag("concept_bank_headings", json.dumps(concept_bank.get("headings", [])[:24], indent=2))
    concept_keywords = _tag("concept_bank_keywords", json.dumps(concept_bank.get("keywords", [])[:40], indent=2))
    concept_summary = _tag("concept_bank_summary", concept_bank.get("summary", ""))
    spec_block = _tag("project_spec", spec)
    current_block = _tag("current_tasks", json.dumps(current_tasks, indent=2))
    memory_block = _tag("memory_packet", memory_digest or "No recent memory traces.")
    ordered_block = _tag("ordered_planning_procedure", "\n".join(f"{i+1}. {step}" for i, step in enumerate(ORDERED_PLANNING_PROCEDURE)))
    response_contract = _tag(
        "response_contract",
        "\n".join([
            "Wrap the final answer as [response][json]{...}[/json][/response].",
            "Do not emit prose outside the response envelope.",
            "Use dependency titles or base task IDs in dependencies.",
            "Prefer coherent, resource-grounded tasks over generic feature requests.",
            "Bias toward memory-rich, verification-aware, repair-ready task packs.",
        ]),
    )
    json_schema_block = _tag(
        "json_schema",
        json.dumps({
            "mode_hint": "reasoning | anomaly_detection | maintenance | delivery",
            "task_ideas": [
                {
                    "title": "string",
                    "lane": "architecture|backend|frontend|infra|test|docs",
                    "description": "string",
                    "dependencies": ["T001 or task title"],
                    "resources": ["path"],
                    "priority": 0.0,
                    "uncertainty": 0.0,
                    "color_hint": {"gold": 0.1, "emerald": 0.1, "amber": 0.1, "violet": 0.1, "crimson": 0.1, "white": 0.1, "cyan": 0.1},
                    "rationale": "string",
                    "rare_condition": False,
                    "learning_value": 0.0
                }
            ],
            "election_criteria": {
                "goal_alignment": 0.0,
                "lane_coverage": 0.0,
                "learning_density": 0.0,
                "repair_readiness": 0.0,
                "rare_condition_readiness": 0.0,
                "memory_advancement": 0.0,
                "operator_visibility": 0.0
            }
        }, indent=2)
    )
    prefill = _tag("response_prefill", "[response]\n[json]")
    prompt = "\n\n".join([
        ADVANCED_RUNTIME_TASK_SYSTEM_PROMPT,
        concept_block,
        _tag("memory_advancement_brief", MEMORY_ADVANCEMENT_GUIDE_TEXT),
        spec_block,
        concept_headings,
        concept_keywords,
        concept_summary,
        current_block,
        memory_block,
        ordered_block,
        build_ordered_action_script(max_tasks=max_tasks),
        response_contract,
        json_schema_block,
        prefill,
    ]).strip()
    return prompt

def heuristic_task_pack(state: PlannerState) -> Dict[str, Any]:
    ideas = [
        StarlingTaskIdea(
            title="Create MAPF reservation lattice",
            lane="architecture",
            description="Define the repository control surface, lane boundaries, reservation windows, and resource-occupancy rules that keep agents from colliding on files, contracts, or handoffs.",
            dependencies=["T001"],
            resources=["contracts/mapf_reservation_lattice.json", "docs/control_surface.md"],
            priority=0.91,
            uncertainty=0.18,
            color_hint={"gold": 0.20, "emerald": 0.16, "amber": 0.14, "white": 0.10},
            rationale="Safe execution begins with an explicit control surface and reservation grammar.",
            rare_condition=False,
            learning_value=0.78,
        ),
        StarlingTaskIdea(
            title="Build agent handoff election matrix",
            lane="backend",
            description="Implement handoff scoring that combines lane fit, reservation pressure, palette compatibility, and verification readiness so runtime execution can elect safer owners task by task.",
            dependencies=["Create MAPF reservation lattice"],
            resources=["src/runtime/handoff_election.py", "contracts/handoff_election_matrix.json"],
            priority=0.90,
            uncertainty=0.22,
            color_hint={"gold": 0.17, "white": 0.16, "emerald": 0.14, "amber": 0.10},
            rationale="The runtime needs a reusable election surface, not ad hoc reassignment.",
            rare_condition=False,
            learning_value=0.74,
        ),
        StarlingTaskIdea(
            title="Add PennyLane circuit orchestration pack",
            lane="backend",
            description="Materialize the intent, task pigment, scheduler, memory, and reset evaluators as pluggable circuit interfaces with a classical surrogate fallback and a shared scoring contract.",
            dependencies=["Build agent handoff election matrix"],
            resources=["src/runtime/quantum_pack.py", "contracts/quantum_pack_contract.json"],
            priority=0.92,
            uncertainty=0.24,
            color_hint={"violet": 0.18, "gold": 0.18, "emerald": 0.12, "cyan": 0.12},
            rationale="Quantum scoring must be a first-class orchestration layer.",
            rare_condition=False,
            learning_value=0.83,
        ),
        StarlingTaskIdea(
            title="Create chromatic palette state registry",
            lane="architecture",
            description="Define how Gold, Emerald, Amber, Violet, Crimson, White, and Cyan interact as decision surfaces, including how palette mixes affect urgency, caution, and resolution.",
            dependencies=["Add PennyLane circuit orchestration pack"],
            resources=["contracts/chromatic_palette_registry.json", "docs/chromatic_palette_registry.md"],
            priority=0.88,
            uncertainty=0.20,
            color_hint={"gold": 0.18, "cyan": 0.18, "violet": 0.16, "white": 0.12},
            rationale="Color cognition needs a canonical registry so tasks, agents, and circuits share semantics.",
            rare_condition=False,
            learning_value=0.76,
        ),
        StarlingTaskIdea(
            title="Implement episodic memory braid",
            lane="backend",
            description="Fuse resets, failures, repairs, colorsync broadcasts, plan elections, and completions into a long-horizon replay braid with stable summarization and calibration hooks.",
            dependencies=["Add PennyLane circuit orchestration pack"],
            resources=["src/runtime/episodic_memory_braid.py", "contracts/episodic_memory_braid.json"],
            priority=0.94,
            uncertainty=0.23,
            color_hint={"emerald": 0.16, "white": 0.18, "violet": 0.12, "cyan": 0.10},
            rationale="The runtime should remember execution as structured episodes, not disconnected trace lines.",
            rare_condition=False,
            learning_value=0.95,
        ),
        StarlingTaskIdea(
            title="Create rare-condition anomaly witness capsules",
            lane="test",
            description="Capture anomaly preconditions, palette deltas, reset pressure, and post-repair evidence into reusable witness capsules that future elections can consult.",
            dependencies=["Implement episodic memory braid", "Create chromatic palette state registry"],
            resources=["src/runtime/anomaly_capsules.py", "tests/test_anomaly_capsules.py"],
            priority=0.93,
            uncertainty=0.26,
            color_hint={"crimson": 0.18, "amber": 0.18, "white": 0.12, "violet": 0.12},
            rationale="Rare discoveries become valuable only when captured with enough evidence to replay safely.",
            rare_condition=True,
            learning_value=0.96,
        ),
        StarlingTaskIdea(
            title="Add ColorSync mentor quorum replay",
            lane="backend",
            description="Require at least one teacher and multiple student observations before a rare palette delta becomes shared policy, with trust weighting and anti-herding safeguards.",
            dependencies=["Implement episodic memory braid", "Create rare-condition anomaly witness capsules"],
            resources=["src/runtime/mentor_quorum_replay.py", "contracts/mentor_quorum_replay.json"],
            priority=0.92,
            uncertainty=0.27,
            color_hint={"cyan": 0.18, "violet": 0.16, "gold": 0.16, "white": 0.10},
            rationale="Cross-agent learning should use quorum replay so one noisy outlier does not dominate the fleet.",
            rare_condition=True,
            learning_value=0.98,
        ),
        StarlingTaskIdea(
            title="Create palette rumor quarantine",
            lane="test",
            description="Build an outlier tribunal that can quarantine suspicious palette broadcasts until verification, witness capsules, and mentor quorum signals agree.",
            dependencies=["Add ColorSync mentor quorum replay"],
            resources=["src/runtime/palette_quarantine.py", "tests/test_palette_quarantine.py"],
            priority=0.90,
            uncertainty=0.28,
            color_hint={"amber": 0.20, "crimson": 0.14, "white": 0.14, "violet": 0.14},
            rationale="ColorSync needs a containment mechanism for bad contagion and false alarms.",
            rare_condition=True,
            learning_value=0.89,
        ),
        StarlingTaskIdea(
            title="Build stability ladder verification gate",
            lane="test",
            description="Implement a verify -> repair -> reverify ladder that checks syntax, contracts, workspace integrity, operator artifacts, and runtime observability before closure.",
            dependencies=["Create palette rumor quarantine"],
            resources=["src/runtime/stability_ladder.py", "tests/test_stability_ladder.py"],
            priority=0.95,
            uncertainty=0.18,
            color_hint={"white": 0.20, "amber": 0.18, "emerald": 0.16, "crimson": 0.08},
            rationale="The runtime should not declare success without passing a staged assurance ladder.",
            rare_condition=False,
            learning_value=0.81,
        ),
        StarlingTaskIdea(
            title="Add repair synthesizer and rollback diary",
            lane="backend",
            description="Generate repair actions, patch proposals, rollback notes, and post-repair evidence so failed tasks become safe recovery loops instead of one-off retries.",
            dependencies=["Build stability ladder verification gate"],
            resources=["src/runtime/repair_synthesizer.py", "docs/rollback_diary.md"],
            priority=0.94,
            uncertainty=0.22,
            color_hint={"white": 0.18, "amber": 0.16, "crimson": 0.14, "gold": 0.10},
            rationale="Repairs should be explicit planner outputs with evidence and reversibility.",
            rare_condition=False,
            learning_value=0.86,
        ),
        StarlingTaskIdea(
            title="Create counterfactual archive and plan-memory tuner",
            lane="docs",
            description="Record rejected plans, near-miss elections, and failed patches so later runs can tune weights and prompt emphasis using more than only the winning path.",
            dependencies=["Add repair synthesizer and rollback diary"],
            resources=["reports/counterfactual_archive.json", "docs/plan_memory_tuner.md"],
            priority=0.89,
            uncertainty=0.21,
            color_hint={"violet": 0.18, "white": 0.16, "gold": 0.14, "cyan": 0.10},
            rationale="Learning from rejected branches improves future elections and prompt targeting.",
            rare_condition=False,
            learning_value=0.92,
        ),
        StarlingTaskIdea(
            title="Publish operator observability atlas",
            lane="docs",
            description="Ship an operator-facing atlas for election rationale, anomaly alerts, mentor edges, explanation graphs, and memory braid summaries.",
            dependencies=["Create counterfactual archive and plan-memory tuner"],
            resources=["docs/operator_observability_atlas.md", "reports/explanation_graph.json"],
            priority=0.87,
            uncertainty=0.16,
            color_hint={"white": 0.22, "gold": 0.16, "violet": 0.12},
            rationale="Humans need strong visibility into why the runtime chose this plan and what it learned.",
            rare_condition=False,
            learning_value=0.68,
        ),
    ]
    return {
        "mode_hint": "reasoning",
        "task_ideas": [asdict(x) for x in ideas],
        "election_criteria": {
            "goal_alignment": 0.95,
            "lane_coverage": 0.90,
            "learning_density": 0.97,
            "repair_readiness": 0.93,
            "rare_condition_readiness": 0.95,
            "memory_advancement": 0.98,
            "operator_visibility": 0.89,
        },
    }

def maybe_generate_starling_task_pack(prompt: str) -> Tuple[str, Dict[str, Any]]:
    text = llm_complete(prompt)
    payload = extract_json_block(text)
    if payload:
        payload.setdefault("_raw_response", text[:4000])
        return "starling", payload
    return "heuristic", {}

ADVANCED_PROMPT_PREVIEW = build_advanced_task_prompt(
    spec=DEFAULT_SPEC,
    state=_preview_state,
    memory_digest=build_memory_prompt_packet(MemoryStore(WORKSPACE_ROOT.parent / "agentic_runtime_prompt_preview.db"), limit=12, concept_script_items=CONCEPT_SCRIPT_ITEMS),
    concept_bank=concept_bank,
    max_tasks=14,
)
print(ADVANCED_PROMPT_PREVIEW[:3200] + "\n\n... [truncated preview] ...")


In [ ]:

# @title 6) Advanced main election + huge runtime execution cell
# This cell intentionally contains the full orchestrated runtime.

@dataclass
class PlanGenome:
    name: str
    weight_scale: float = 1.0
    reset_bias: float = 0.0
    colorsync_bias: float = 0.0
    teaching_trust: float = 0.22
    anomaly_bias: float = 0.0
    max_steps: int = 11
    generated_task_budget: int = 16

@dataclass
class EpisodeOutcome:
    genome: str
    workspace: str
    score: float
    completed_count: int
    failed_count: int
    verification_score: float
    smoke_ok: bool
    colorsync_events: int
    reset_events: int
    mode_hint: str

ADVANCED_STAGE_ORDER = ["discovery", "coordination", "intelligence", "assurance", "operator"]


def palette_from_task(task: TaskNode) -> ColorVector:
    base = {name: 0.08 for name in COLOR_ORDER}
    if getattr(task, "color_hint", None):
        for k, v in task.color_hint.items():
            if k in base:
                base[k] += float(v)
    lane_bias = {
        "architecture": {"gold": 0.06, "violet": 0.04},
        "backend": {"emerald": 0.06, "amber": 0.04},
        "frontend": {"cyan": 0.06, "gold": 0.04},
        "infra": {"amber": 0.06, "crimson": 0.04},
        "test": {"amber": 0.07, "white": 0.05},
        "docs": {"white": 0.06, "gold": 0.03},
    }
    for k, v in lane_bias.get(task.lane, {}).items():
        base[k] = base.get(k, 0.0) + v
    if task.rare_condition:
        base["crimson"] += 0.08
        base["amber"] += 0.05
    return ColorVector.from_dict(base)

def reservation_pressure(state: PlannerState, task: TaskNode, step: int) -> float:
    if not state.reservations:
        return 0.0
    overlap = 0
    for r in state.reservations:
        if not (r.start_step <= step <= r.end_step):
            continue
        if r.resource in set(task.resources):
            overlap += 1
    return clamp(overlap / max(len(task.resources), 1))

def reserve_task_resources(state: PlannerState, task: TaskNode, agent_name: str, step: int) -> None:
    for resource in list(task.resources) or [f"virtual:{task.task_id}"]:
        state.reservations.append(Reservation(resource=resource, task_id=task.task_id, agent_name=agent_name, start_step=step, end_step=step + 1))

def colorsync_update(agent_palette: ColorVector, observed_palette: ColorVector, trust: float = 0.3) -> ColorVector:
    return agent_palette.blend(observed_palette, alpha=clamp(trust, 0.05, 0.95))



def memory_bias_vector(memory_digest: str) -> ColorVector:
    lower = (memory_digest or "").lower()
    bias = {"gold": 0.10, "emerald": 0.10, "amber": 0.10, "violet": 0.10, "crimson": 0.10, "white": 0.10, "cyan": 0.10}
    if "repair" in lower:
        bias["white"] += 0.08
        bias["amber"] += 0.06
    if "reset" in lower:
        bias["crimson"] += 0.06
        bias["amber"] += 0.04
    if "colorsync" in lower or "teach" in lower:
        bias["cyan"] += 0.07
        bias["violet"] += 0.04
    if "complete" in lower:
        bias["emerald"] += 0.08
        bias["gold"] += 0.04
    return ColorVector.from_dict(bias)


def task_execution_probability(
    task: TaskNode,
    agent: ChromaticAgentState,
    qpack: QuantumCircuitPack,
    weights: WeightProfile,
    state: PlannerState,
    step: int,
    memory_digest: str,
) -> Tuple[float, Dict[str, Any]]:
    task_palette = palette_from_task(task)
    memory_bias = memory_bias_vector(memory_digest)
    blended = task_palette.blend(memory_bias, alpha=weights.memory_blend_alpha)
    intent = qpack.intent_metrics(
        urgency=task.priority,
        value=max(task.priority, 1.0 - task.uncertainty),
        certainty=1.0 - task.uncertainty,
        coordination_pressure=reservation_pressure(state, task, step),
    )
    pigment = qpack.task_pigment_metrics(agent.palette.blend(blended, alpha=0.5))
    memory = qpack.memory_metrics(
        recency=0.65 if state.completed else 0.24,
        stability=1.0 - min(1.0, len(state.failed) / max(len(state.completed) + len(state.failed), 1)),
        novelty=0.72 if task.source != "base" else 0.42,
        repair_success=0.70 if "repair" in memory_digest.lower() else 0.30,
    )
    lane_fit = agent.lane_match(task)
    compatibility = agent.palette.similarity(blended)
    score = (
        intent["intent_score"] * weights.intent +
        pigment["task_score"] * weights.task_pigment +
        memory["memory_score"] * weights.memory +
        lane_fit * 0.12 +
        compatibility * 0.12
    )
    return clamp(score), {
        "intent": intent,
        "pigment": pigment,
        "memory": memory,
        "lane_fit": lane_fit,
        "compatibility": compatibility,
    }

def select_scheduler_mode(state: PlannerState, weights: WeightProfile, qpack: QuantumCircuitPack) -> Dict[str, Any]:
    maintenance = clamp(0.34 + len(state.failed) / max(len(state.tasks), 1))
    reasoning = clamp(0.42 + len(state.ready_tasks()) / max(len(state.tasks), 1))
    anomaly = clamp(len([t for t in state.tasks.values() if t.rare_condition]) / max(len(state.tasks), 1) + len(state.failed) * 0.08)
    delivery = clamp(len(state.completed) / max(len(state.tasks), 1))
    return qpack.scheduler_metrics(maintenance, reasoning, anomaly, delivery)


def _task_to_payload_idea(task: TaskNode) -> Dict[str, Any]:
    return {
        "title": task.title,
        "lane": task.lane,
        "description": task.description,
        "dependencies": task.dependencies,
        "resources": task.resources,
        "priority": task.priority,
        "uncertainty": task.uncertainty,
        "color_hint": task.color_hint,
        "rationale": task.rationale,
        "rare_condition": task.rare_condition,
    }

def build_advanced_runtime_tasks(start_index: int = 700) -> List[TaskNode]:
    specs = [
        {
            "title": "Materialize stage-gated execution graph",
            "lane": "architecture",
            "description": "Define discovery, coordination, intelligence, assurance, and operator stages so the runtime can advance safely through explicit control surfaces.",
            "dependencies": ["T001", "T003"],
            "resources": ["contracts/stage_gate_schema.json", "docs/stage_gate_execution.md"],
            "priority": 0.90,
            "uncertainty": 0.18,
            "color_hint": {"gold": 0.22, "white": 0.16, "amber": 0.10, "emerald": 0.12},
            "rationale": "A stage-gated runtime is easier to inspect and safer to evolve.",
            "rare_condition": False,
        },
        {
            "title": "Build operator cockpit and palette telemetry surface",
            "lane": "docs",
            "description": "Emit operator-facing status, palette snapshots, plan election rows, verification outcomes, and episode summaries into a compact observability cockpit.",
            "dependencies": ["T004", "T006"],
            "resources": ["docs/operator_cockpit.md", "reports/operator_dashboard.json"],
            "priority": 0.86,
            "uncertainty": 0.20,
            "color_hint": {"white": 0.22, "gold": 0.14, "emerald": 0.12},
            "rationale": "Operators need a high-signal view of the runtime.",
            "rare_condition": False,
        },
        {
            "title": "Create ColorSync mentorship ledger",
            "lane": "architecture",
            "description": "Specify how agents learn from other agents by replaying palette deltas, trust shifts, and rare-condition discoveries without collapsing into herd behavior.",
            "dependencies": ["T003", "T004"],
            "resources": ["contracts/colorsync_mentorship_ledger.json", "docs/colorsync_mentorship.md"],
            "priority": 0.92,
            "uncertainty": 0.22,
            "color_hint": {"cyan": 0.20, "gold": 0.18, "violet": 0.16, "white": 0.08},
            "rationale": "Cross-agent teaching must be first-class, not accidental.",
            "rare_condition": False,
        },
        {
            "title": "Author rare-condition sentinel probes",
            "lane": "backend",
            "description": "Generate probes that look for palette outliers, reservation contention, reset cascades, and verification drift before those conditions become system-wide failures.",
            "dependencies": ["T002", "T003", "T004"],
            "resources": ["src/runtime/sentinels.py", "tests/test_sentinels.py"],
            "priority": 0.91,
            "uncertainty": 0.26,
            "color_hint": {"crimson": 0.22, "amber": 0.18, "violet": 0.12, "white": 0.10},
            "rationale": "Anomaly probes should intervene before unstable behavior spreads.",
            "rare_condition": True,
        },
        {
            "title": "Run replay calibration tournament",
            "lane": "backend",
            "description": "Use memory traces and prior episode summaries to calibrate weight profiles, reset thresholds, ColorSync thresholds, and stabilization strength.",
            "dependencies": ["T002", "T004"],
            "resources": ["src/runtime/calibration.py", "reports/calibration_profile.json"],
            "priority": 0.89,
            "uncertainty": 0.21,
            "color_hint": {"emerald": 0.18, "violet": 0.14, "gold": 0.12, "white": 0.10},
            "rationale": "The runtime should learn from its own traces.",
            "rare_condition": False,
        },
        {
            "title": "Scan contract drift and verification envelopes",
            "lane": "test",
            "description": "Track whether emitted contracts, scheduler claims, and runtime artifacts remain aligned with the generated plan and repository surface.",
            "dependencies": ["T005"],
            "resources": ["contracts/verification_envelope.json", "tests/test_verification_envelope.py"],
            "priority": 0.88,
            "uncertainty": 0.18,
            "color_hint": {"amber": 0.22, "white": 0.18, "emerald": 0.12},
            "rationale": "Verification should cover emitted artifacts and contracts, not just syntax.",
            "rare_condition": False,
        },
        {
            "title": "Prepare repair-readiness patch pack",
            "lane": "backend",
            "description": "Pre-bake fallback patches, recovery stubs, and minimal replacements so the runtime can recover quickly after verification or smoke-test failures.",
            "dependencies": ["T005"],
            "resources": ["src/runtime/repair_pack.py", "docs/repair_readiness.md"],
            "priority": 0.87,
            "uncertainty": 0.23,
            "color_hint": {"white": 0.20, "amber": 0.18, "crimson": 0.10},
            "rationale": "Repair loops work better when recovery materials already exist.",
            "rare_condition": True,
        },
        {
            "title": "Compile agent-to-agent palette teaching casebook",
            "lane": "docs",
            "description": "Capture examples where one agent discovered a rare condition and other agents updated their palettes, trust scores, or routing choices as a result.",
            "dependencies": ["T004", "T006"],
            "resources": ["docs/colorsync_casebook.md", "reports/colorsync_teaching_log.json"],
            "priority": 0.86,
            "uncertainty": 0.20,
            "color_hint": {"cyan": 0.18, "white": 0.18, "gold": 0.12},
            "rationale": "The system should preserve examples of useful agent-to-agent learning.",
            "rare_condition": False,
        },
        {
            "title": "Run counterfactual plan arena and champion election",
            "lane": "architecture",
            "description": "Execute multiple weight and policy variants, compare bundles, and elect a champion execution profile rather than trusting a single run.",
            "dependencies": ["T001", "T002", "T004"],
            "resources": ["reports/arena_report.json", "docs/champion_election.md"],
            "priority": 0.93,
            "uncertainty": 0.24,
            "color_hint": {"gold": 0.22, "violet": 0.18, "white": 0.12, "cyan": 0.12},
            "rationale": "Episode-level election makes the runtime more robust.",
            "rare_condition": False,
        },
    ]
    tasks = []
    cursor = start_index
    for spec in specs:
        cursor += 1
        tasks.append(
            TaskNode(
                task_id=f"T{cursor:03d}",
                title=spec["title"],
                lane=spec["lane"],
                description=spec["description"],
                dependencies=list(spec["dependencies"]),
                resources=list(spec["resources"]),
                priority=float(spec["priority"]),
                uncertainty=float(spec["uncertainty"]),
                source="advanced_runtime",
                color_hint=dict(spec["color_hint"]),
                rationale=spec["rationale"],
                rare_condition=bool(spec["rare_condition"]),
            )
        )
    return tasks

def dedupe_tasks(tasks: List[TaskNode]) -> List[TaskNode]:
    seen = set()
    out = []
    for task in tasks:
        key = (task.title.strip().lower(), task.lane, tuple(sorted(task.resources)))
        if key in seen:
            continue
        seen.add(key)
        out.append(task)
    return out

def stage_for_task(task: TaskNode) -> str:
    text = f"{task.title} {task.description}".lower()
    if any(x in text for x in ["architecture", "contract", "lane structure", "control surface", "stage-gated", "champion election"]):
        return "discovery"
    if any(x in text for x in ["scheduler", "reservation", "handoff", "mentorship", "colorsync"]):
        return "coordination"
    if any(x in text for x in ["quantum", "palette", "memory", "calibration", "sentinel", "anomaly", "replay"]):
        return "intelligence"
    if any(x in text for x in ["verify", "verification", "repair", "drift", "envelope", "patch"]):
        return "assurance"
    return "operator"

def build_stage_gate_index(tasks: Iterable[TaskNode]) -> Dict[str, str]:
    return {task.task_id: stage_for_task(task) for task in tasks}

def stage_gate_ready(task: TaskNode, state: PlannerState, gate_index: Dict[str, str]) -> bool:
    my_gate = gate_index.get(task.task_id, "operator")
    my_idx = ADVANCED_STAGE_ORDER.index(my_gate)
    if my_idx == 0:
        return True
    earlier_gates = set(ADVANCED_STAGE_ORDER[:my_idx])
    earlier = [t for t in state.tasks.values() if gate_index.get(t.task_id) in earlier_gates]
    if not earlier:
        return True
    return all(t.status in {"completed", "failed"} for t in earlier if t.task_id != task.task_id)

def mutate_weights(base: WeightProfile, genome: PlanGenome) -> WeightProfile:
    return WeightProfile(
        intent=round(clamp(base.intent * genome.weight_scale, 0.16, 0.40), 6),
        task_pigment=round(clamp(base.task_pigment * genome.weight_scale, 0.20, 0.42), 6),
        memory=round(clamp(base.memory * (0.96 + genome.anomaly_bias * 0.05), 0.12, 0.30), 6),
        scheduler=round(clamp(base.scheduler * genome.weight_scale, 0.12, 0.32), 6),
        colorsync=round(clamp(base.colorsync + genome.colorsync_bias, 0.08, 0.22), 6),
        reset_threshold=round(clamp(base.reset_threshold + genome.reset_bias, 0.52, 0.84), 6),
        colorsync_threshold=round(clamp(base.colorsync_threshold - genome.colorsync_bias * 0.3, 0.10, 0.28), 6),
        stabilization_alpha=round(clamp(base.stabilization_alpha + genome.reset_bias * 0.4, 0.08, 0.28), 6),
        memory_blend_alpha=round(clamp(base.memory_blend_alpha + genome.anomaly_bias * 0.03, 0.10, 0.28), 6),
    )

def anomaly_forecast(task: TaskNode, agent: ChromaticAgentState, state: PlannerState, step: int, qpack: QuantumCircuitPack) -> Dict[str, Any]:
    pressure = reservation_pressure(state, task, step)
    rarity = 0.75 if task.rare_condition else 0.25
    crimson = palette_from_task(task).to_dict().get("crimson", 0.0)
    metrics = qpack.reset_metrics(
        ambiguity=clamp(0.20 + task.uncertainty + rarity * 0.10),
        overload=clamp(pressure + len(state.failed) / max(len(state.tasks), 1)),
        conflict=clamp(crimson + pressure * 0.25),
        drift=clamp(0.22 + (1.0 - min(1.0, len(state.completed) / max(len(state.tasks), 1)))),
    )
    score = float(metrics["reset_score"] * 0.6 + pressure * 0.2 + rarity * 0.2)
    return {"score": round(clamp(score), 6), "metrics": metrics, "pressure": pressure}

def choose_agent_for_task(task: TaskNode, agents: List[ChromaticAgentState], qpack: QuantumCircuitPack, weights: WeightProfile, state: PlannerState, step: int, memory_digest: str, genome: PlanGenome) -> Tuple[Optional[ChromaticAgentState], Dict[str, Any]]:
    rows = []
    for agent in agents:
        probability, details = task_execution_probability(task, agent, qpack, weights, state, step, memory_digest)
        forecast = anomaly_forecast(task, agent, state, step, qpack)
        score = probability * (1.0 + 0.05 * genome.weight_scale) - forecast["score"] * (0.25 + genome.anomaly_bias * 0.15)
        score += agent.trust.get(agent.profile.name, 0.0) if agent.profile.name in agent.trust else 0.0
        rows.append((score, probability, forecast, agent, details))
    rows.sort(key=lambda x: x[0], reverse=True)
    if not rows:
        return None, {}
    score, probability, forecast, agent, details = rows[0]
    payload = dict(details)
    payload.update({
        "selection_score": round(float(score), 6),
        "probability": round(float(probability), 6),
        "forecast": forecast,
    })
    return agent, payload

def create_recovery_task(task: TaskNode, counter: int) -> TaskNode:
    return TaskNode(
        task_id=f"F{counter:03d}",
        title=f"Recover {task.title}",
        lane="backend" if task.lane != "docs" else "docs",
        description=f"Recovery path for {task.title}: emit minimal valid scaffold and operator note so downstream tasks can continue safely.",
        dependencies=[],
        resources=list(task.resources[:2]) or ["docs/recovery_note.md"],
        priority=clamp(task.priority * 0.92 + 0.04),
        uncertainty=clamp(task.uncertainty * 0.75),
        source="recovery",
        color_hint={"white": 0.22, "amber": 0.18, "emerald": 0.10},
        rationale="Recovery tasks make failures survivable.",
        rare_condition=False,
    )

def emit_advanced_artifacts(
    workspace: Path,
    episode_bundle: Dict[str, Any],
    operator_dashboard: Dict[str, Any],
    teaching_log: List[Dict[str, Any]],
    anomaly_log: List[Dict[str, Any]],
    gate_index: Dict[str, str],
) -> None:
    (workspace / "reports").mkdir(parents=True, exist_ok=True)
    (workspace / "docs").mkdir(parents=True, exist_ok=True)
    (workspace / "contracts").mkdir(parents=True, exist_ok=True)
    (workspace / "reports/operator_dashboard.json").write_text(json.dumps(operator_dashboard, indent=2))
    (workspace / "reports/colorsync_teaching_log.json").write_text(json.dumps(teaching_log, indent=2))
    (workspace / "reports/anomaly_forecast_log.json").write_text(json.dumps(anomaly_log, indent=2))
    (workspace / "contracts/stage_gate_index.json").write_text(json.dumps(gate_index, indent=2))
    arena_md = textwrap.dedent(f"""
    # Advanced Runtime Arena Report

    - Champion genome: **{episode_bundle.get("champion_genome", "")}**
    - Backend: **{episode_bundle.get("backend", "")}**
    - Mode hint: **{episode_bundle.get("mode_hint", "")}**
    - Completed tasks: **{len(episode_bundle.get("completed", []))}**
    - Failed tasks: **{len(episode_bundle.get("failed", []))}**
    - ColorSync events: **{len(episode_bundle.get("colorsync_events", []))}**
    - Reset events: **{len(episode_bundle.get("reset_events", []))}**

    ## Stage gates
    {json.dumps(episode_bundle.get("gate_summary", {}), indent=2)}

    ## Operator cockpit keys
    {json.dumps(sorted(operator_dashboard.keys()), indent=2)}
    """).strip() + "\n"
    (workspace / "docs/advanced_runtime_arena.md").write_text(arena_md)

    test_path = workspace / "tests" / "test_advanced_runtime.py"
    test_path.parent.mkdir(parents=True, exist_ok=True)
    test_code = textwrap.dedent("""
    import json
    from pathlib import Path

    def test_operator_dashboard_exists():
        path = Path("reports/operator_dashboard.json")
        data = json.loads(path.read_text())
        assert "mode_hint" in data
        assert "completed_count" in data

    def test_stage_gate_index_exists():
        path = Path("contracts/stage_gate_index.json")
        data = json.loads(path.read_text())
        assert isinstance(data, dict)
        assert len(data) > 0
    """).strip() + "\n"
    test_path.write_text(test_code)



def local_plan_election(
    candidates: List[Tuple[str, Dict[str, Any]]],
    state: PlannerState,
    concept_bank: Dict[str, Any],
    memory_digest: str,
    qpack: QuantumCircuitPack,
) -> Dict[str, Any]:
    rows = []
    keyword_set = set(concept_bank.get("keywords", []))
    for idx, (source_name, payload) in enumerate(candidates):
        mode_hint, tasks, election_criteria = parse_task_pack(payload, start_index=120 + idx * 40)
        lane_cov = len(set(t.lane for t in tasks)) / max(len(LANE_ORDER), 1)
        learn_density = sum(1 for t in tasks if "colorsync" in t.title.lower() or "learn" in t.description.lower() or "teach" in t.title.lower()) / max(len(tasks), 1)
        rare_ready = sum(1 for t in tasks if t.rare_condition) / max(len(tasks), 1)
        repair_ready = sum(1 for t in tasks if "repair" in t.title.lower() or "verify" in t.title.lower()) / max(len(tasks), 1)
        concept_hits = 0
        for task in tasks:
            text = f"{task.title} {task.description} {task.rationale}".lower()
            concept_hits += sum(1 for kw in list(keyword_set)[:18] if kw.lower() in text)
        concept_align = clamp(concept_hits / max(len(tasks) * 3, 1))
        palette_energy = 0.0
        for task in tasks:
            palette_energy += np.mean(list(palette_from_task(task).to_dict().values()))
        palette_energy = clamp(palette_energy / max(len(tasks), 1))
        scheduler = qpack.scheduler_metrics(
            maintenance=0.30 + repair_ready * 0.30,
            reasoning=0.38 + lane_cov * 0.20,
            anomaly=0.20 + rare_ready * 0.35,
            delivery=0.18 + concept_align * 0.22,
        )
        score = (
            lane_cov * 0.22 +
            learn_density * 0.18 +
            rare_ready * 0.12 +
            repair_ready * 0.16 +
            concept_align * 0.22 +
            palette_energy * 0.10
        )
        rows.append({
            "source": source_name,
            "mode_hint": mode_hint,
            "task_count": len(tasks),
            "lane_coverage": round(lane_cov, 6),
            "learning_density": round(learn_density, 6),
            "rare_readiness": round(rare_ready, 6),
            "repair_readiness": round(repair_ready, 6),
            "concept_alignment": round(concept_align, 6),
            "palette_energy": round(palette_energy, 6),
            "scheduler_hint": scheduler.get("mode", "reasoning"),
            "score": round(score, 6),
            "tasks": tasks,
            "payload": payload,
            "election_criteria": election_criteria,
        })
    rows = sorted(rows, key=lambda r: r["score"], reverse=True)
    champion = rows[0] if rows else {"source": "none", "mode_hint": "reasoning", "tasks": [], "payload": {}, "score": 0.0}
    return {"champion": champion, "rows": rows}




def scaffold_contents_for_path(rel_path: str) -> str:
    rel = str(rel_path)
    if rel.endswith(".py"):
        return textwrap.dedent(f"""
        \"\"\"Auto-generated scaffold for {rel}.\"\"\"

        def runtime_status() -> str:
            return "ok"
        """).strip() + "\n"
    if rel.endswith(".json"):
        return json.dumps({"path": rel, "status": "scaffolded"}, indent=2) + "\n"
    if rel.endswith(".md"):
        return f"# {Path(rel).stem.replace('_', ' ').title()}\n\nGenerated scaffold for `{rel}`.\n"
    if rel.endswith(".txt"):
        return f"generated scaffold for {rel}\n"
    return f"generated scaffold for {rel}\n"

def write_workspace_scaffold(state: PlannerState, workspace: Path) -> Dict[str, str]:
    if workspace.exists():
        shutil.rmtree(workspace)
    workspace.mkdir(parents=True, exist_ok=True)
    files = {}
    for task in state.tasks.values():
        for rel in task.resources:
            path = workspace / rel
            path.parent.mkdir(parents=True, exist_ok=True)
            content = scaffold_contents_for_path(rel)
            path.write_text(content)
            files[rel] = content
    return files

def emit_runtime_files(workspace: Path, state: PlannerState, prompt_text: str, election: Dict[str, Any], run_bundle: Dict[str, Any]) -> None:
    for dirname in ["src/runtime", "tests", "docs", "contracts", "reports"]:
        (workspace / dirname).mkdir(parents=True, exist_ok=True)
    (workspace / "src/runtime/__init__.py").write_text("")
    (workspace / "src/runtime/core.py").write_text(textwrap.dedent("""
    def runtime_status() -> str:
        return "ok"

    def colorsync_status() -> str:
        return "stable"
    """).strip() + "\n")
    (workspace / "tests/test_runtime_smoke.py").write_text(textwrap.dedent("""
    from src.runtime.core import runtime_status, colorsync_status

    def test_runtime_status():
        assert runtime_status() == "ok"
        assert colorsync_status() == "stable"
    """).strip() + "\n")
    (workspace / "docs/starling_advanced_task_prompt.txt").write_text(prompt_text)
    rows = [{k: v for k, v in row.items() if k not in {"tasks", "payload"}} for row in election.get("rows", [])]
    (workspace / "reports/plan_election.json").write_text(json.dumps({"rows": rows, "champion_source": election.get("champion", {}).get("source", "")}, indent=2))
    (workspace / "docs/generated_task_pack.json").write_text(json.dumps({
        "tasks": [asdict(t) for t in state.tasks.values()],
        "run_bundle": run_bundle,
    }, indent=2, default=str))

def verify_workspace(workspace: Path) -> Dict[str, Any]:
    py_files = [str(p) for p in workspace.rglob("*.py")]
    syntax_errors = []
    for path in py_files:
        try:
            ast.parse(Path(path).read_text(), filename=path)
        except SyntaxError as e:
            syntax_errors.append({"path": path, "detail": str(e)})
    contract_paths = [str(p) for p in (workspace / "contracts").glob("*.json")] if (workspace / "contracts").exists() else []
    docs_present = (workspace / "docs").exists()
    compile_ok = compileall.compile_dir(str(workspace), quiet=1)
    return {
        "python_file_count": len(py_files),
        "syntax_error_count": len(syntax_errors),
        "syntax_errors": syntax_errors,
        "contract_count": len(contract_paths),
        "docs_present": docs_present,
        "compileall_ok": bool(compile_ok),
    }

def synthesize_repairs(workspace: Path, verification: Dict[str, Any]) -> List[TaskNode]:
    repairs = []
    if not verification["docs_present"]:
        repairs.append(TaskNode(
            task_id="R001",
            title="Repair missing docs",
            lane="docs",
            description="Add missing operator documentation.",
            resources=["README.md", "docs/operator_workflow.md"],
            priority=0.88,
            uncertainty=0.14,
            source="repair",
            color_hint={"white": 0.22, "amber": 0.16},
            rationale="Docs should exist for operator visibility.",
        ))
    if verification["syntax_error_count"] > 0:
        repairs.append(TaskNode(
            task_id="R002",
            title="Repair syntax drift",
            lane="backend",
            description="Overwrite broken python files with valid stubs.",
            resources=[err["path"].replace(str(workspace) + "/", "") for err in verification["syntax_errors"]],
            priority=0.92,
            uncertainty=0.16,
            source="repair",
            color_hint={"white": 0.18, "amber": 0.18, "crimson": 0.10},
            rationale="Syntax drift must be repaired before release.",
            rare_condition=True,
        ))
    return repairs

def apply_repairs(workspace: Path, repairs: List[TaskNode]) -> None:
    for repair in repairs:
        for rel in repair.resources:
            path = workspace / rel
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(scaffold_contents_for_path(rel))

def run_smoke_tests(workspace: Path) -> Dict[str, Any]:
    tests_dir = workspace / "tests"
    if not tests_dir.exists():
        return {"returncode": 0, "stdout": "", "stderr": "", "ran": False}
    cmd = [sys.executable, "-m", "pytest", "-q", str(tests_dir)]
    try:
        proc = subprocess.run(cmd, cwd=str(workspace), capture_output=True, text=True, timeout=60)
        return {"returncode": proc.returncode, "stdout": proc.stdout[-2000:], "stderr": proc.stderr[-2000:], "ran": True}
    except Exception as e:
        return {"returncode": 1, "stdout": "", "stderr": str(e), "ran": False}


def main_election(spec: str, enable_starling: bool = False, generated_task_budget: int = 16) -> Dict[str, Any]:
    election_state = PlannerState(spec=spec)
    election_state.add_tasks(deterministic_seed_tasks(spec))
    memory = MemoryStore(WORKSPACE_ROOT.parent / "agentic_runtime_v12_election.db")
    memory.write_trace("election", None, "concept_script_loaded", {"count": len(CONCEPT_SCRIPT_ITEMS)})
    memory_digest = build_memory_prompt_packet(memory, limit=60, concept_script_items=CONCEPT_SCRIPT_ITEMS)
    prompt_text = build_advanced_task_prompt(spec=spec, state=election_state, memory_digest=memory_digest, concept_bank=concept_bank, max_tasks=generated_task_budget)
    memory.write_trace("election", None, "prompt_built", {"chars": len(prompt_text), "tagged_actions": True, "memory_rich": True})

    candidates: List[Tuple[str, Dict[str, Any]]] = [("heuristic", heuristic_task_pack(election_state))]
    if enable_starling:
        source_name, llm_payload = maybe_generate_starling_task_pack(prompt_text)
        if llm_payload:
            candidates.append((source_name, llm_payload))

    advanced_tasks = build_advanced_runtime_tasks(start_index=700)
    advanced_payload = {
        "mode_hint": "reasoning",
        "task_ideas": [_task_to_payload_idea(task) for task in advanced_tasks],
        "election_criteria": {
            "goal": "promote stage-gated execution, agent learning, anomaly readiness, and operator visibility",
            "prefer": ["colorsync learning", "memory traces", "verification", "repair readiness", "observability"],
        },
    }
    candidates.append(("advanced_runtime", advanced_payload))
    qpack = QuantumCircuitPack()
    election = local_plan_election(candidates, election_state, concept_bank, memory_digest, qpack)
    return {
        "prompt_text": prompt_text,
        "election": election,
        "advanced_tasks": advanced_tasks,
        "concept_bank": concept_bank,
    }

def execute_episode(spec: str, election_bundle: Dict[str, Any], genome: PlanGenome, episode_index: int) -> Dict[str, Any]:
    workspace = WORKSPACE_ROOT.parent / f"{WORKSPACE_ROOT.name}_v12_episode_{episode_index}_{genome.name.lower()}"
    memory = MemoryStore(WORKSPACE_ROOT.parent / f"{WORKSPACE_ROOT.name}_v12_episode_{episode_index}_{genome.name.lower()}.db")
    qpack = QuantumCircuitPack()
    calibrator = QuantumCircuitCalibrator()

    base_tasks = deterministic_seed_tasks(spec)
    champion_tasks = list(election_bundle["election"]["champion"]["tasks"])
    advanced_tasks = build_advanced_runtime_tasks(start_index=760)
    all_tasks = dedupe_tasks(base_tasks + champion_tasks + advanced_tasks)

    state = PlannerState(spec=spec)
    state.add_tasks(all_tasks)
    gate_index = build_stage_gate_index(state.tasks.values())
    agents = bootstrap_agents()

    memory.write_trace("episode", None, "episode_started", {"genome": genome.name, "episode_index": episode_index})
    for item in CONCEPT_SCRIPT_ITEMS:
        memory.write_trace("concept", None, "concept_loaded", {"name": item["name"], "purpose": item["purpose"]})

    base_weights = calibrator.calibrate(memory.recent_traces(limit=48))
    weights = mutate_weights(base_weights, genome)

    assignments = []
    colorsync_events = []
    reset_events = []
    anomaly_log = []
    teaching_log = []
    step_logs = []
    memory_digest = build_memory_prompt_packet(memory, limit=80, concept_script_items=CONCEPT_SCRIPT_ITEMS)
    recovery_counter = 900

    for step in range(genome.max_steps):
        scheduler = select_scheduler_mode(state, weights, qpack)
        ready = [t for t in state.ready_tasks() if stage_gate_ready(t, state, gate_index)]
        if not ready:
            memory.write_trace("step", None, "no_ready_tasks", {"step": step})
            if step > 2:
                break
            continue

        ready = sorted(ready, key=lambda t: (-t.priority, t.uncertainty, t.task_id))
        for task in ready[:4]:
            agent, scored = choose_agent_for_task(task, agents, qpack, weights, state, step, memory_digest, genome)
            if agent is None:
                continue

            anomaly = scored.get("forecast", {})
            anomaly_log.append({
                "step": step,
                "task_id": task.task_id,
                "task": task.title,
                "agent": agent.profile.name,
                "forecast": anomaly,
            })

            if anomaly.get("score", 0.0) >= max(0.62, weights.reset_threshold - 0.02):
                reset_events.append({
                    "step": step,
                    "task_id": task.task_id,
                    "agent": agent.profile.name,
                    "reason": "anomaly threshold crossed",
                    "score": anomaly.get("score", 0.0),
                })
                memory.write_trace("reset", task, "reset", {"score": anomaly.get("score", 0.0), "agent": agent.profile.name})
                agent.palette = agent.palette.blend(memory_bias_vector(memory_digest), alpha=weights.stabilization_alpha)

            reserve_task_resources(state, task, agent.profile.name, step)
            task.status = "running"
            memory.write_trace("step", task, "task_claimed", {"agent": agent.profile.name, "mode": scheduler["mode"], "selection_score": scored.get("selection_score")})

            det = int(stable_hash(f"{genome.name}:{task.task_id}:{step}:{agent.profile.name}"), 16) % 1000 / 1000.0
            probability = clamp(scored.get("probability", 0.5))
            if "verify" in task.title.lower() or "repair" in task.title.lower():
                probability = clamp(probability + 0.08)
            if task.rare_condition:
                probability = clamp(probability - 0.06 + genome.anomaly_bias * 0.04)

            if det <= probability:
                task.status = "completed"
                state.completed.append(task.task_id)
                agent.completed.append(task.task_id)
                memory.write_trace("step", task, "task_completed", {"agent": agent.profile.name, "probability": probability})
                if task.rare_condition or "colorsync" in task.title.lower() or "palette" in task.title.lower():
                    broadcast = PaletteBroadcast(
                        source_agent=agent.profile.name,
                        task_id=task.task_id,
                        confidence=round(probability, 6),
                        palette=agent.palette.to_dict(),
                        rare_condition=bool(task.rare_condition),
                        note=f"{task.title} updated the shared palette field.",
                    )
                    state.broadcasts.append(broadcast)
                    colorsync_events.append(asdict(broadcast))
                    memory.write_trace("colorsync", task, "colorsync", asdict(broadcast))
                    for other in agents:
                        if other.profile.name == agent.profile.name:
                            continue
                        trust = clamp(other.trust.get(agent.profile.name, genome.teaching_trust) + (0.05 if task.rare_condition else 0.02), 0.12, 0.60)
                        other.trust[agent.profile.name] = trust
                        before = other.palette.to_dict()
                        other.palette = colorsync_update(other.palette, agent.palette, trust=trust * 0.45 + weights.colorsync)
                        teaching_log.append({
                            "step": step,
                            "teacher": agent.profile.name,
                            "student": other.profile.name,
                            "task_id": task.task_id,
                            "rare_condition": task.rare_condition,
                            "trust": round(trust, 6),
                            "before": before,
                            "after": other.palette.to_dict(),
                        })
                memory_digest = build_memory_prompt_packet(memory, limit=80, concept_script_items=CONCEPT_SCRIPT_ITEMS)
            else:
                task.status = "failed"
                state.failed.append(task.task_id)
                memory.write_trace("step", task, "task_failed", {"agent": agent.profile.name, "probability": probability, "det": det})
                recovery_counter += 1
                recovery_task = create_recovery_task(task, recovery_counter)
                if recovery_task.task_id not in state.tasks:
                    state.add_task(recovery_task)

        step_logs.append({
            "step": step,
            "mode": scheduler["mode"],
            "ready_count": len(ready),
            "completed_total": len(state.completed),
            "failed_total": len(state.failed),
            "reset_events_total": len(reset_events),
            "colorsync_events_total": len(colorsync_events),
        })
        if len(state.completed) >= min(12, max(8, len(state.tasks) // 2)):
            break

    scaffold = write_workspace_scaffold(state, workspace)
    emit_runtime_files(
        workspace=workspace,
        state=state,
        prompt_text=election_bundle["prompt_text"],
        election=election_bundle["election"],
        run_bundle={
            "backend": qpack.backend,
            "mode_hint": election_bundle["election"]["champion"]["mode_hint"],
            "completed": state.completed,
            "failed": state.failed,
            "colorsync_events": colorsync_events,
            "reset_events": reset_events,
            "ordered_planning_procedure": ORDERED_PLANNING_PROCEDURE,
            "concept_script_items": CONCEPT_SCRIPT_ITEMS,
            "genome": asdict(genome),
        },
    )

    verification_before = verify_workspace(workspace)
    repairs = synthesize_repairs(workspace, verification_before)
    for repair in repairs:
        memory.write_trace("repair", repair, "repair_synthesized", {"detail": repair.description})
    if repairs:
        apply_repairs(workspace, repairs)
    verification_after = verify_workspace(workspace)
    smoke = run_smoke_tests(workspace)

    gate_summary = {}
    for gate in ADVANCED_STAGE_ORDER:
        gate_tasks = [t for t in state.tasks.values() if gate_index.get(t.task_id) == gate]
        gate_summary[gate] = {
            "count": len(gate_tasks),
            "completed": sum(1 for t in gate_tasks if t.status == "completed"),
            "failed": sum(1 for t in gate_tasks if t.status == "failed"),
            "pending": sum(1 for t in gate_tasks if t.status == "pending"),
        }

    verification_score = (
        (1.0 if verification_after["syntax_error_count"] == 0 else 0.2) * 0.45 +
        (1.0 if verification_after["docs_present"] else 0.2) * 0.20 +
        (1.0 if verification_after["compileall_ok"] else 0.2) * 0.20 +
        clamp(verification_after["contract_count"] / 8.0) * 0.15
    )
    smoke_ok = smoke["returncode"] == 0
    score = (
        clamp(len(state.completed) / max(len(state.completed) + len(state.failed), 1)) * 0.46 +
        verification_score * 0.26 +
        (0.12 if smoke_ok else 0.0) +
        clamp(len(colorsync_events) / 4.0) * 0.10 +
        clamp(len(teaching_log) / 8.0) * 0.06
    )

    operator_dashboard = {
        "workspace": str(workspace),
        "backend": qpack.backend,
        "mode_hint": election_bundle["election"]["champion"]["mode_hint"],
        "completed_count": len(state.completed),
        "failed_count": len(state.failed),
        "verification_after": verification_after,
        "smoke_ok": smoke_ok,
        "colorsync_event_count": len(colorsync_events),
        "teaching_event_count": len(teaching_log),
        "reset_event_count": len(reset_events),
        "gate_summary": gate_summary,
        "genome": asdict(genome),
    }

    episode_bundle = {
        "workspace": str(workspace),
        "backend": qpack.backend,
        "champion_genome": genome.name,
        "mode_hint": election_bundle["election"]["champion"]["mode_hint"],
        "completed": state.completed,
        "failed": state.failed,
        "assignments": assignments,
        "colorsync_events": colorsync_events,
        "reset_events": reset_events,
        "verification_before": verification_before,
        "verification_after": verification_after,
        "smoke_tests": smoke,
        "weights": asdict(weights),
        "gate_summary": gate_summary,
        "step_logs": step_logs,
        "anomaly_log_sample": anomaly_log[:20],
        "teaching_log_sample": teaching_log[:20],
        "prompt_text": election_bundle["prompt_text"][:6000],
        "concept_script_items": CONCEPT_SCRIPT_ITEMS,
        "ordered_planning_procedure": ORDERED_PLANNING_PROCEDURE,
    }

    emit_advanced_artifacts(workspace, episode_bundle, operator_dashboard, teaching_log, anomaly_log, gate_index)

    return {
        "workspace": str(workspace),
        "backend": qpack.backend,
        "mode_hint": election_bundle["election"]["champion"]["mode_hint"],
        "completed_count": len(state.completed),
        "failed_count": len(state.failed),
        "assignment_count": len(assignments),
        "colorsync_events": colorsync_events,
        "reset_events": reset_events,
        "verification_before": verification_before,
        "verification_after": verification_after,
        "smoke_tests": smoke,
        "prompt_text": election_bundle["prompt_text"],
        "concept_script_text": CONCEPT_SCRIPT_TEXT,
        "bundle": episode_bundle,
        "score": round(score, 6),
        "operator_dashboard": operator_dashboard,
        "teaching_log": teaching_log,
        "anomaly_log": anomaly_log,
        "outcome": asdict(EpisodeOutcome(
            genome=genome.name,
            workspace=str(workspace),
            score=round(score, 6),
            completed_count=len(state.completed),
            failed_count=len(state.failed),
            verification_score=round(verification_score, 6),
            smoke_ok=smoke_ok,
            colorsync_events=len(colorsync_events),
            reset_events=len(reset_events),
            mode_hint=election_bundle["election"]["champion"]["mode_hint"],
        )),
    }

def main_execution(spec: str, enable_starling: bool = False, max_steps: int = 11, generated_task_budget: int = 16) -> Dict[str, Any]:
    election_bundle = main_election(spec, enable_starling=enable_starling, generated_task_budget=generated_task_budget)
    genomes = [
        PlanGenome(name="Balanced", weight_scale=1.00, reset_bias=0.00, colorsync_bias=0.00, teaching_trust=0.22, anomaly_bias=0.00, max_steps=max_steps, generated_task_budget=generated_task_budget),
        PlanGenome(name="Explorer", weight_scale=1.05, reset_bias=-0.02, colorsync_bias=0.03, teaching_trust=0.26, anomaly_bias=0.03, max_steps=max_steps + 1, generated_task_budget=generated_task_budget),
        PlanGenome(name="Guardian", weight_scale=0.97, reset_bias=0.04, colorsync_bias=-0.01, teaching_trust=0.18, anomaly_bias=0.06, max_steps=max_steps, generated_task_budget=generated_task_budget),
    ]
    episodes = [execute_episode(spec, election_bundle, genome, idx + 1) for idx, genome in enumerate(genomes)]
    episodes_sorted = sorted(episodes, key=lambda item: item["score"], reverse=True)
    champion = episodes_sorted[0]

    champion["bundle"]["arena"] = {
        "episode_count": len(episodes),
        "episodes": [ep["outcome"] for ep in episodes],
        "winner": champion["outcome"]["genome"],
    }
    champion["bundle"]["plan_election"] = [
        {k: v for k, v in row.items() if k not in {"tasks", "payload"}}
        for row in election_bundle["election"]["rows"]
    ]
    champion["bundle"]["operator_dashboard"] = champion["operator_dashboard"]

    runtime_summary_path = Path(champion["workspace"]) / "docs" / "advanced_runtime_bundle.json"
    runtime_summary_path.parent.mkdir(parents=True, exist_ok=True)
    runtime_summary_path.write_text(json.dumps(champion["bundle"], indent=2))

    return {
        "workspace": champion["workspace"],
        "backend": champion["backend"],
        "mode_hint": champion["mode_hint"],
        "completed_count": champion["completed_count"],
        "failed_count": champion["failed_count"],
        "assignment_count": champion["assignment_count"],
        "colorsync_events": champion["colorsync_events"],
        "reset_events": champion["reset_events"],
        "verification_before": champion["verification_before"],
        "verification_after": champion["verification_after"],
        "smoke_tests": champion["smoke_tests"],
        "bundle": champion["bundle"],
        "prompt_text": champion["prompt_text"],
        "concept_script_text": champion["concept_script_text"],
        "score": champion["score"],
        "operator_dashboard": champion["operator_dashboard"],
        "teaching_log": champion["teaching_log"],
        "anomaly_log": champion["anomaly_log"],
    }


# -------- v13 advanced orchestration overrides --------
ADVANCED_WORKSPACE_ROOT = Path("/tmp/agentic_workspace_v13")


def render_concept_script_text() -> str:
    blocks = []
    for idx, item in enumerate(CONCEPT_SCRIPT_ITEMS, start=1):
        musts = "\n".join(f"  - {x}" for x in item.get("must_show_up_as_tasks", []))
        blocks.append(textwrap.dedent(f"""
        {idx}. {item['name']}
           purpose: {item['purpose']}
           operator_question: {item['operator_question']}
           must_show_up_as_tasks:
        {musts}
        """).strip())
    return "\n\n".join(blocks)


CONCEPT_SCRIPT_TEXT = render_concept_script_text()


class StrategyArchive:
    def __init__(self):
        self.rows: List[Dict[str, Any]] = []

    def add(self, row: Dict[str, Any]) -> None:
        self.rows.append(dict(row))

    def champion(self) -> Dict[str, Any]:
        if not self.rows:
            return {}
        return max(self.rows, key=lambda r: float(r.get("score", 0.0)))


@dataclass
class MentorEdge:
    mentor: str
    learner: str
    trust: float
    reason: str


@dataclass
class OperatorAlert:
    level: str
    code: str
    message: str
    step: int
    task_id: Optional[str] = None


def deterministic_ratio(*parts: Any) -> float:
    text = "|".join(str(p) for p in parts)
    return int(stable_hash(text), 16) / float(int("f" * 12, 16))


def task_semantic_flags(task: TaskNode) -> Dict[str, bool]:
    text = f"{task.title} {task.description} {task.rationale}".lower()
    return {
        "verification": any(x in text for x in ["verify", "verification", "contract", "test", "check"]),
        "repair": any(x in text for x in ["repair", "recover", "rollback", "patch"]),
        "colorsync": any(x in text for x in ["colorsync", "palette", "mentor", "learning from other agents", "teach"]),
        "memory": any(x in text for x in ["memory", "trace", "replay", "calibration"]),
        "scheduling": any(x in text for x in ["mapf", "reservation", "handoff", "schedule", "conflict"]),
        "quantum": any(x in text for x in ["quantum", "pennylane", "circuit", "pigment", "intent"]),
        "operator": any(x in text for x in ["operator", "artifact", "observability", "dashboard", "visibility"]),
        "rare": bool(task.rare_condition) or "rare" in text or "anomaly" in text,
    }


def lane_coverage_score(tasks: List[TaskNode]) -> float:
    lanes = {t.lane for t in tasks}
    return round(len(lanes) / max(len(LANE_ORDER), 1), 6)


def concept_coverage_score(tasks: List[TaskNode]) -> Dict[str, float]:
    scores = {"verification": 0, "repair": 0, "colorsync": 0, "memory": 0, "scheduling": 0, "quantum": 0, "operator": 0, "rare": 0}
    for t in tasks:
        flags = task_semantic_flags(t)
        for key, present in flags.items():
            if present:
                scores[key] += 1
    n = max(len(tasks), 1)
    return {k: round(v / n, 6) for k, v in scores.items()}


def build_mentor_graph(agents: List[ChromaticAgentState]) -> List[MentorEdge]:
    edges: List[MentorEdge] = []
    for agent in agents:
        other_rows = []
        for other in agents:
            if other.profile.name == agent.profile.name:
                continue
            similarity = agent.palette.similarity(other.palette)
            trust = clamp(0.45 * similarity + 0.55 * other.trust.get(agent.profile.name, 0.12))
            other_rows.append((trust, other))
        other_rows.sort(key=lambda x: x[0], reverse=True)
        if other_rows:
            trust, mentor = other_rows[0]
            edges.append(MentorEdge(mentor=mentor.profile.name, learner=agent.profile.name, trust=round(trust, 6), reason="palette alignment + trust"))
    return edges


def apply_broadcast_learning(agents: List[ChromaticAgentState], broadcast: PaletteBroadcast, trust_floor: float = 0.12) -> List[Dict[str, Any]]:
    teaching_events = []
    observed = ColorVector.from_dict(broadcast.palette)
    for learner in agents:
        if learner.profile.name == broadcast.source_agent:
            continue
        mentor_bonus = learner.trust.get(broadcast.source_agent, trust_floor)
        alpha = clamp(0.14 + mentor_bonus * 0.5 + (0.08 if broadcast.rare_condition else 0.0), 0.08, 0.68)
        before = learner.palette.to_dict()
        learner.palette = colorsync_update(learner.palette, observed, trust=alpha)
        learner.trust[broadcast.source_agent] = round(clamp(learner.trust.get(broadcast.source_agent, trust_floor) + 0.05), 6)
        teaching_events.append({
            "mentor": broadcast.source_agent,
            "learner": learner.profile.name,
            "alpha": round(alpha, 6),
            "rare_condition": broadcast.rare_condition,
            "before": before,
            "after": learner.palette.to_dict(),
            "note": broadcast.note,
        })
    return teaching_events


def build_followup_verification_task(task: TaskNode, counter: int) -> TaskNode:
    return TaskNode(
        task_id=f"V{counter:03d}",
        title=f"Verify {task.title}",
        lane="test",
        description=f"Verification gate for {task.title}: syntax, contracts, generated files, and observability expectations.",
        dependencies=[task.task_id],
        resources=["tests/test_generated_runtime.py", "contracts/generated_contracts.json", "docs/verification_gate.md"],
        priority=clamp(task.priority * 0.95 + 0.04),
        uncertainty=clamp(task.uncertainty * 0.78),
        source="verification",
        color_hint={"white": 0.24, "amber": 0.18, "emerald": 0.08},
        rationale="Every substantive runtime change should have a follow-up verification gate.",
        rare_condition=False,
    )


def build_counterfactual_probe(task: TaskNode, counter: int) -> TaskNode:
    return TaskNode(
        task_id=f"C{counter:03d}",
        title=f"Counterfactual probe for {task.title}",
        lane="backend",
        description=f"Simulate alternative execution paths for {task.title} before rollout and record whether the plan should branch, delay, or continue.",
        dependencies=[task.task_id],
        resources=["src/runtime/counterfactual_probe.py", "reports/counterfactual_probe.json"],
        priority=clamp(task.priority * 0.84 + 0.05),
        uncertainty=clamp(task.uncertainty * 0.82),
        source="counterfactual",
        color_hint={"violet": 0.18, "cyan": 0.14, "amber": 0.12, "white": 0.08},
        rationale="The runtime should model alternate futures instead of only reacting after failures.",
        rare_condition=task.rare_condition,
    )


def ensure_advanced_control_surface(tasks: List[TaskNode]) -> List[TaskNode]:
    existing_titles = {t.title for t in tasks}
    injected: List[TaskNode] = []
    required_specs = [
        {
            "title": "Create trust-weighted ColorSync mentor lattice",
            "lane": "architecture",
            "description": "Formalize mentor selection, trust updates, anti-herding safeguards, and replay rules for agent palette learning.",
            "dependencies": ["T001"],
            "resources": ["contracts/mentor_lattice.json", "docs/colorsync_mentors.md"],
            "priority": 0.91,
            "uncertainty": 0.22,
            "color_hint": {"cyan": 0.18, "gold": 0.14, "violet": 0.12, "white": 0.08},
            "rationale": "Cross-agent learning needs first-class structure.",
            "rare_condition": False,
        },
        {
            "title": "Add counterfactual execution arena",
            "lane": "backend",
            "description": "Score plan branches before execution and keep a compact champion archive with why one route won over another.",
            "dependencies": ["T002", "T003"],
            "resources": ["src/runtime/counterfactual_arena.py", "reports/champion_archive.json"],
            "priority": 0.88,
            "uncertainty": 0.24,
            "color_hint": {"violet": 0.16, "amber": 0.12, "white": 0.10, "gold": 0.08},
            "rationale": "A more advanced runtime should compare branches, not just execute a single line blindly.",
            "rare_condition": False,
        },
        {
            "title": "Emit explanation graph and operator escalations",
            "lane": "docs",
            "description": "Publish a graph of stage transitions, failures, recoveries, mentor broadcasts, and alerts for operator review.",
            "dependencies": ["T004", "T006"],
            "resources": ["reports/explanation_graph.json", "reports/operator_alerts.json", "docs/escalation_playbook.md"],
            "priority": 0.86,
            "uncertainty": 0.2,
            "color_hint": {"white": 0.18, "emerald": 0.12, "amber": 0.10},
            "rationale": "Operators need a coherent story of what the runtime decided and why.",
            "rare_condition": False,
        },
    ]
    cursor = 980
    for spec in required_specs:
        if spec["title"] in existing_titles:
            continue
        cursor += 1
        injected.append(TaskNode(task_id=f"A{cursor}", source="advanced_injected", **spec))
    return dedupe_tasks(tasks + injected)


def counterfactual_plan_score(tasks: List[TaskNode], genome: PlanGenome, qpack: QuantumCircuitPack, memory_digest: str, concept_bank: Dict[str, Any]) -> Dict[str, Any]:
    coverage = concept_coverage_score(tasks)
    lane_score = lane_coverage_score(tasks)
    rare_count = sum(1 for t in tasks if t.rare_condition)
    learning_count = sum(1 for t in tasks if task_semantic_flags(t)["colorsync"])
    observability_count = sum(1 for t in tasks if task_semantic_flags(t)["operator"])
    avg_priority = statistics.mean([t.priority for t in tasks]) if tasks else 0.0
    avg_uncertainty = statistics.mean([t.uncertainty for t in tasks]) if tasks else 1.0
    memory_vec = memory_bias_vector(memory_digest)
    palette_mix = memory_vec.blend(ColorVector.from_dict({"gold": 0.17, "emerald": 0.14, "amber": 0.12, "violet": 0.16, "crimson": 0.10, "white": 0.17, "cyan": 0.14}), alpha=0.35)
    q_metrics = qpack.task_pigment_metrics(palette_mix)
    intent = qpack.intent_metrics(urgency=avg_priority, value=avg_priority, certainty=1.0 - avg_uncertainty, coordination_pressure=0.35)
    score = 0.16 * lane_score
    score += 0.11 * coverage["scheduling"]
    score += 0.11 * coverage["memory"]
    score += 0.12 * coverage["verification"]
    score += 0.10 * coverage["repair"]
    score += 0.11 * coverage["quantum"]
    score += 0.11 * coverage["operator"]
    score += 0.12 * min(learning_count / 2.0, 1.0)
    score += 0.04 * min(rare_count / 2.0, 1.0)
    score += 0.02 * min(observability_count / 2.0, 1.0)
    score += 0.05 * float(q_metrics["task_score"])
    score += 0.05 * float(intent["intent_score"])
    score *= (1.0 + 0.03 * genome.weight_scale)
    score -= 0.06 * max(avg_uncertainty - 0.35, 0.0)
    score += 0.02 * min(len(concept_bank.get("keywords", [])) / 24.0, 1.0)
    return {
        "score": round(clamp(score, 0.0, 1.0), 6),
        "coverage": coverage,
        "lane_score": lane_score,
        "rare_count": rare_count,
        "learning_count": learning_count,
        "observability_count": observability_count,
        "q_task_score": q_metrics["task_score"],
        "intent_score": intent["intent_score"],
    }


def build_explanation_graph(step_logs: List[Dict[str, Any]], colorsync_events: List[Dict[str, Any]], repair_tasks: List[TaskNode], alerts: List[OperatorAlert]) -> Dict[str, Any]:
    nodes = []
    edges = []
    for row in step_logs:
        node_id = f"step:{row['step']}:{row['task_id']}"
        nodes.append({"id": node_id, "type": row.get("event", "task"), "label": row.get("task", row['task_id']), "agent": row.get("agent")})
        if row.get("depends_on"):
            for dep in row["depends_on"]:
                edges.append({"source": dep, "target": node_id, "type": "dependency"})
    for event in colorsync_events:
        event_id = f"colorsync:{event['step']}:{event['task_id']}"
        nodes.append({"id": event_id, "type": "colorsync", "label": event['task'], "agent": event['source_agent']})
        for learner in event.get("learners", []):
            edges.append({"source": event_id, "target": f"agent:{learner}", "type": "teaches"})
    for task in repair_tasks:
        nodes.append({"id": f"repair:{task.task_id}", "type": "repair", "label": task.title})
    for alert in alerts:
        nodes.append({"id": f"alert:{alert.step}:{alert.code}", "type": "alert", "label": alert.message})
    return {"nodes": nodes, "edges": edges}


def emit_super_artifacts(workspace: Path, run_bundle: Dict[str, Any], mentor_edges: List[MentorEdge], explanation_graph: Dict[str, Any], alerts: List[OperatorAlert], archive: StrategyArchive) -> None:
    (workspace / "reports").mkdir(parents=True, exist_ok=True)
    (workspace / "docs").mkdir(parents=True, exist_ok=True)
    (workspace / "contracts").mkdir(parents=True, exist_ok=True)
    (workspace / "reports/mentor_graph.json").write_text(json.dumps([asdict(x) for x in mentor_edges], indent=2))
    (workspace / "reports/explanation_graph.json").write_text(json.dumps(explanation_graph, indent=2))
    (workspace / "reports/operator_alerts.json").write_text(json.dumps([asdict(x) for x in alerts], indent=2))
    (workspace / "contracts/strategy_archive.json").write_text(json.dumps(archive.rows, indent=2))
    champion = archive.champion()
    champion_md = textwrap.dedent(f"""
    # Champion Archive

    - champion genome: **{champion.get('genome', '')}**
    - champion score: **{champion.get('score', 0.0)}**
    - completed_count: **{champion.get('completed_count', 0)}**
    - failed_count: **{champion.get('failed_count', 0)}**
    - mode_hint: **{champion.get('mode_hint', '')}**
    """).strip() + "\n"
    (workspace / "docs/champion_archive.md").write_text(champion_md)
    (workspace / "docs/concept_script_expanded.txt").write_text(CONCEPT_SCRIPT_TEXT)
    (workspace / "reports/final_runtime_bundle.json").write_text(json.dumps(run_bundle, indent=2))


def local_plan_election(candidates: List[Tuple[str, Dict[str, Any]]], state: PlannerState, concept_bank: Dict[str, Any], memory_digest: str, qpack: QuantumCircuitPack) -> Dict[str, Any]:
    genomes = [
        PlanGenome(name="Balanced", weight_scale=1.0, reset_bias=0.02, colorsync_bias=0.04, teaching_trust=0.24, anomaly_bias=0.04, max_steps=11, generated_task_budget=18),
        PlanGenome(name="Explorer", weight_scale=1.08, reset_bias=-0.02, colorsync_bias=0.10, teaching_trust=0.30, anomaly_bias=0.02, max_steps=12, generated_task_budget=18),
        PlanGenome(name="Guardian", weight_scale=0.95, reset_bias=0.08, colorsync_bias=-0.02, teaching_trust=0.20, anomaly_bias=0.10, max_steps=10, generated_task_budget=16),
    ]
    rows = []
    for source_name, payload in candidates:
        mode_hint, tasks, election_meta = parse_task_pack(payload, start_index=100 + len(rows) * 30)
        tasks = ensure_advanced_control_surface(tasks)
        if not tasks:
            continue
        rollout_rows = []
        for genome in genomes:
            metrics = counterfactual_plan_score(tasks, genome, qpack, memory_digest, concept_bank)
            rollout_rows.append({"genome": genome.name, **metrics})
        champion_rollout = max(rollout_rows, key=lambda r: r["score"])
        row = {
            "source": source_name,
            "mode_hint": mode_hint,
            "score": champion_rollout["score"],
            "rollouts": rollout_rows,
            "champion_rollout": champion_rollout,
            "coverage": champion_rollout["coverage"],
            "lane_score": champion_rollout["lane_score"],
            "task_count": len(tasks),
            "tasks": tasks,
            "payload": payload,
            "election_meta": election_meta,
        }
        rows.append(row)
    rows.sort(key=lambda r: r["score"], reverse=True)
    champion = rows[0] if rows else {"source": "none", "tasks": [], "score": 0.0, "payload": {}, "mode_hint": "reasoning", "champion_rollout": {"genome": "Balanced"}}
    return {"rows": rows, "champion": champion}


def main_election(spec: str, enable_starling: bool = False, generated_task_budget: int = 18) -> Dict[str, Any]:
    election_state = PlannerState(spec=spec)
    election_state.add_tasks(deterministic_seed_tasks(spec))
    concept_bank = load_concept_bank()
    memory = MemoryStore(ADVANCED_WORKSPACE_ROOT.parent / "agentic_runtime_v13_election.db")
    for item in CONCEPT_SCRIPT_ITEMS:
        memory.write_trace("concept", None, "concept_script_loaded", {"name": item["name"], "purpose": item["purpose"]})
    memory_digest = memory.digest(limit=28)
    prompt_text = build_advanced_task_prompt(spec=spec, state=election_state, memory_digest=memory_digest, concept_bank=concept_bank, max_tasks=generated_task_budget)
    memory.write_trace("election", None, "prompt_built", {"chars": len(prompt_text), "tagged_actions": True, "memory_rich": True})

    candidates: List[Tuple[str, Dict[str, Any]]] = [("heuristic", heuristic_task_pack(election_state))]
    if enable_starling:
        source_name, llm_payload = maybe_generate_starling_task_pack(prompt_text)
        if llm_payload:
            candidates.append((source_name, llm_payload))

    advanced_tasks = build_advanced_runtime_tasks(start_index=700)
    advanced_payload = {
        "mode_hint": "reasoning",
        "task_ideas": [_task_to_payload_idea(task) for task in ensure_advanced_control_surface(advanced_tasks)],
        "election_criteria": {
            "goal": "maximize safe execution depth, repair readiness, operator visibility, mentorship density, and counterfactual foresight",
            "prefer": ["verification", "repair", "ColorSync learning", "counterfactual arena", "operator alerts"],
        },
    }
    candidates.append(("advanced_runtime", advanced_payload))

    qpack = QuantumCircuitPack()
    election = local_plan_election(candidates, election_state, concept_bank, memory_digest, qpack)
    return {
        "prompt_text": prompt_text,
        "election": election,
        "advanced_tasks": advanced_tasks,
        "concept_bank": concept_bank,
        "concept_script_text": CONCEPT_SCRIPT_TEXT,
    }


def execute_episode(spec: str, election_bundle: Dict[str, Any], genome: PlanGenome, episode_index: int) -> Dict[str, Any]:
    workspace = ADVANCED_WORKSPACE_ROOT.parent / f"{ADVANCED_WORKSPACE_ROOT.name}_episode_{episode_index}_{genome.name.lower()}"
    memory = MemoryStore(ADVANCED_WORKSPACE_ROOT.parent / f"{ADVANCED_WORKSPACE_ROOT.name}_episode_{episode_index}_{genome.name.lower()}.db")
    qpack = QuantumCircuitPack()
    calibrator = QuantumCircuitCalibrator()

    base_tasks = deterministic_seed_tasks(spec)
    champion_tasks = list(election_bundle["election"]["champion"]["tasks"])
    advanced_tasks = build_advanced_runtime_tasks(start_index=760)
    all_tasks = ensure_advanced_control_surface(dedupe_tasks(base_tasks + champion_tasks + advanced_tasks))

    state = PlannerState(spec=spec)
    state.add_tasks(all_tasks)
    gate_index = build_stage_gate_index(state.tasks.values())
    agents = bootstrap_agents()
    for agent in agents:
        for other in agents:
            if other.profile.name != agent.profile.name:
                agent.trust[other.profile.name] = round(0.12 + deterministic_ratio(agent.profile.name, other.profile.name) * 0.18, 6)

    mentor_edges = build_mentor_graph(agents)

    memory.write_trace("episode", None, "episode_started", {"genome": genome.name, "episode_index": episode_index})
    for item in CONCEPT_SCRIPT_ITEMS:
        memory.write_trace("concept", None, "concept_loaded", {"name": item["name"], "purpose": item["purpose"]})

    base_weights = calibrator.calibrate(memory.recent_traces(limit=48))
    weights = mutate_weights(base_weights, genome)

    assignments = []
    colorsync_events = []
    reset_events = []
    anomaly_log = []
    teaching_log = []
    step_logs = []
    alerts: List[OperatorAlert] = []
    memory_digest = build_memory_prompt_packet(memory, limit=80, concept_script_items=CONCEPT_SCRIPT_ITEMS)
    recovery_counter = 900
    verify_counter = 950
    counterfactual_counter = 970

    for step in range(genome.max_steps):
        scheduler = select_scheduler_mode(state, weights, qpack)
        ready = [t for t in state.ready_tasks() if stage_gate_ready(t, state, gate_index)]
        if not ready:
            memory.write_trace("step", None, "no_ready_tasks", {"step": step, "scheduler": scheduler["mode"]})
            if step > 3:
                break
            continue

        candidate_rows = []
        for task in ready[:8]:
            flags = task_semantic_flags(task)
            coverage_bonus = 0.02 * sum(float(v) for v in flags.values())
            mentor_bonus = 0.03 if flags["colorsync"] else 0.0
            counterfactual_penalty = 0.05 * task.uncertainty
            agent, scored = choose_agent_for_task(task, agents, qpack, weights, state, step, memory_digest, genome)
            if agent is None:
                continue
            cf_score = scored["selection_score"] + coverage_bonus + mentor_bonus - counterfactual_penalty
            candidate_rows.append((cf_score, task, agent, scored, flags))
        candidate_rows.sort(key=lambda x: x[0], reverse=True)
        batch = candidate_rows[: min(2, max(1, len(candidate_rows)))]

        for cf_score, task, agent, scored, flags in batch:
            reserve_task_resources(state, task, agent.profile.name, step)
            task.status = "running"
            memory.write_trace("step", task, "task_claimed", {"agent": agent.profile.name, "scheduler_mode": scheduler["mode"], "selection_score": scored.get("selection_score", 0.0)})

            anomaly = scored.get("forecast", {})
            anomaly_log.append({
                "step": step,
                "task_id": task.task_id,
                "task": task.title,
                "agent": agent.profile.name,
                "forecast": anomaly,
                "counterfactual_score": round(cf_score, 6),
            })

            if anomaly.get("score", 0.0) >= max(0.62, weights.reset_threshold - 0.02):
                reset_events.append({
                    "step": step,
                    "task_id": task.task_id,
                    "agent": agent.profile.name,
                    "reason": "anomaly threshold crossed",
                    "score": anomaly.get("score", 0.0),
                })
                alerts.append(OperatorAlert(level="warn", code="anomaly_reset", message=f"Reset stabilization triggered for {task.title}", step=step, task_id=task.task_id))
                memory.write_trace("reset", task, "reset", {"score": anomaly.get("score", 0.0), "agent": agent.profile.name})
                agent.palette = agent.palette.blend(memory_bias_vector(memory_digest), alpha=weights.stabilization_alpha)

            success_prob = clamp(0.64 + 0.28 * scored.get("probability", 0.55) + (0.03 if flags["verification"] else 0.0) + (0.02 if flags["repair"] else 0.0) + (0.02 if flags["memory"] else 0.0), 0.55, 0.96)
            roll = deterministic_ratio(step, task.task_id, agent.profile.name, genome.name)
            bootstrap_task = task.task_id in {"T001", "T002", "T003", "T004", "T005", "T006"}
            succeeded = (roll <= success_prob) or (bootstrap_task and step <= 2 and roll <= 0.93)

            step_logs.append({
                "step": step,
                "task_id": task.task_id,
                "task": task.title,
                "agent": agent.profile.name,
                "event": "completed" if succeeded else "failed",
                "depends_on": [f"step:{step-1}:{dep}" for dep in task.dependencies],
            })
            assignments.append({
                "step": step,
                "task_id": task.task_id,
                "task": task.title,
                "agent": agent.profile.name,
                "lane": task.lane,
                "selection": scored,
                "success_prob": round(success_prob, 6),
                "roll": round(roll, 6),
                "event": "completed" if succeeded else "failed",
            })

            if succeeded:
                task.status = "completed"
                if task.task_id not in state.completed:
                    state.completed.append(task.task_id)
                agent.completed.append(task.task_id)
                memory.write_trace("task", task, "completed", {"agent": agent.profile.name, "scheduler_mode": scheduler["mode"], "success_prob": success_prob})

                if flags["rare"] or flags["colorsync"]:
                    broadcast = PaletteBroadcast(
                        source_agent=agent.profile.name,
                        task_id=task.task_id,
                        confidence=round(success_prob, 6),
                        palette=agent.palette.to_dict(),
                        rare_condition=bool(flags["rare"]),
                        note=f"Broadcast from {task.title}",
                    )
                    state.broadcasts.append(broadcast)
                    teachings = apply_broadcast_learning(agents, broadcast, trust_floor=genome.teaching_trust)
                    learners = [t["learner"] for t in teachings]
                    colorsync_events.append({
                        "step": step,
                        "task_id": task.task_id,
                        "task": task.title,
                        "source_agent": agent.profile.name,
                        "rare_condition": bool(flags["rare"]),
                        "learners": learners,
                    })
                    teaching_log.extend(teachings)
                    memory.write_trace("colorsync", task, "broadcast", {"source_agent": agent.profile.name, "learners": learners, "rare_condition": bool(flags["rare"])})

                if task.lane in {"architecture", "backend", "frontend", "infra"} and not flags["verification"]:
                    verify_counter += 1
                    state.add_task(build_followup_verification_task(task, verify_counter))
                if task.uncertainty >= 0.34 and not flags["repair"]:
                    counterfactual_counter += 1
                    state.add_task(build_counterfactual_probe(task, counterfactual_counter))
            else:
                task.status = "failed"
                if task.task_id not in state.failed:
                    state.failed.append(task.task_id)
                memory.write_trace("task", task, "failed", {"agent": agent.profile.name, "scheduler_mode": scheduler["mode"], "success_prob": success_prob})
                recovery_counter += 1
                recovery = create_recovery_task(task, recovery_counter)
                state.add_task(recovery)
                alerts.append(OperatorAlert(level="error", code="task_failed", message=f"Task failed and recovery injected: {task.title}", step=step, task_id=task.task_id))
                if not flags["verification"]:
                    verify_counter += 1
                    state.add_task(build_followup_verification_task(task, verify_counter))

        memory_digest = build_memory_prompt_packet(memory, limit=80, concept_script_items=CONCEPT_SCRIPT_ITEMS)

    scaffold = write_workspace_scaffold(state, workspace)
    run_bundle_seed = {
        "completed": list(state.completed),
        "failed": list(state.failed),
        "assignments": assignments,
        "colorsync_events": colorsync_events,
        "reset_events": reset_events,
        "teaching_log": teaching_log,
        "anomaly_log": anomaly_log,
    }
    emit_runtime_files(workspace, state, election_bundle["prompt_text"], election_bundle["election"], run_bundle_seed)
    verification = verify_workspace(workspace)
    repairs = synthesize_repairs(workspace, verification)
    if verification["syntax_error_count"] > 0 or not verification["docs_present"]:
        alerts.append(OperatorAlert(level="warn", code="verification_repair", message="Verification produced repair tasks", step=len(step_logs), task_id=None))
    apply_repairs(workspace, repairs)
    post_verification = verify_workspace(workspace)
    smoke = run_smoke_tests(workspace)

    operator_dashboard = {
        "workspace": str(workspace),
        "genome": genome.name,
        "backend": qpack.backend,
        "mode_hint": select_scheduler_mode(state, weights, qpack)["mode"],
        "completed_count": len(state.completed),
        "failed_count": len(state.failed),
        "verification": verification,
        "post_verification": post_verification,
        "smoke_tests": smoke,
        "colorsync_events": len(colorsync_events),
        "teaching_events": len(teaching_log),
        "reset_events": len(reset_events),
    }

    explanation_graph = build_explanation_graph(step_logs, colorsync_events, repairs, alerts)
    episode_bundle = {
        "workspace": str(workspace),
        "backend": qpack.backend,
        "mode_hint": operator_dashboard["mode_hint"],
        "completed": state.completed,
        "failed": state.failed,
        "assignments": assignments,
        "verification": verification,
        "post_verification": post_verification,
        "smoke_tests": smoke,
        "champion_genome": genome.name,
        "colorsync_events": colorsync_events,
        "reset_events": reset_events,
        "teaching_log": teaching_log,
        "anomaly_log": anomaly_log,
        "operator_alerts": [asdict(a) for a in alerts],
    }
    emit_advanced_artifacts(workspace, episode_bundle, operator_dashboard, teaching_log, anomaly_log, gate_index)
    emit_super_artifacts(workspace, episode_bundle, mentor_edges, explanation_graph, alerts, StrategyArchive())

    score = 0.36 * min(len(state.completed) / max(len(state.tasks), 1), 1.0)
    score += 0.18 * (1.0 if smoke.get("returncode", 1) == 0 else 0.0)
    score += 0.16 * (1.0 if post_verification.get("syntax_error_count", 1) == 0 else 0.0)
    score += 0.10 * min(len(colorsync_events) / 2.0, 1.0)
    score += 0.08 * min(len(teaching_log) / 3.0, 1.0)
    score += 0.06 * min(len(repairs) / 2.0, 1.0)
    score += 0.06 * min(len(alerts) / 3.0, 1.0)

    return {
        "genome": genome.name,
        "workspace": str(workspace),
        "backend": qpack.backend,
        "mode_hint": operator_dashboard["mode_hint"],
        "score": round(clamp(score, 0.0, 1.0), 6),
        "completed": state.completed,
        "failed": state.failed,
        "completed_count": len(state.completed),
        "failed_count": len(state.failed),
        "verification": verification,
        "post_verification": post_verification,
        "smoke_tests": smoke,
        "colorsync_events": colorsync_events,
        "reset_events": reset_events,
        "teaching_log": teaching_log,
        "anomaly_log": anomaly_log,
        "operator_dashboard": operator_dashboard,
        "operator_alerts": [asdict(a) for a in alerts],
        "mentor_graph": [asdict(x) for x in mentor_edges],
        "bundle": episode_bundle,
        "prompt_text": election_bundle["prompt_text"],
        "concept_script_text": election_bundle["concept_script_text"],
        "explanation_graph": explanation_graph,
    }


def main_execution(spec: str, enable_starling: bool = False, max_steps: int = 12, generated_task_budget: int = 18) -> Dict[str, Any]:
    election_bundle = main_election(spec=spec, enable_starling=enable_starling, generated_task_budget=generated_task_budget)

    genomes = [
        PlanGenome(name="Balanced", weight_scale=1.0, reset_bias=0.02, colorsync_bias=0.04, teaching_trust=0.24, anomaly_bias=0.04, max_steps=max_steps, generated_task_budget=generated_task_budget),
        PlanGenome(name="Mentor", weight_scale=1.02, reset_bias=0.01, colorsync_bias=0.12, teaching_trust=0.32, anomaly_bias=0.05, max_steps=max_steps + 1, generated_task_budget=generated_task_budget),
        PlanGenome(name="Sentinel", weight_scale=0.94, reset_bias=0.10, colorsync_bias=-0.02, teaching_trust=0.18, anomaly_bias=0.12, max_steps=max_steps, generated_task_budget=generated_task_budget - 2),
        PlanGenome(name="Explorer", weight_scale=1.07, reset_bias=-0.02, colorsync_bias=0.10, teaching_trust=0.28, anomaly_bias=0.02, max_steps=max_steps + 1, generated_task_budget=generated_task_budget),
    ]

    archive = StrategyArchive()
    episodes = []
    champion = None
    for idx, genome in enumerate(genomes, start=1):
        result = execute_episode(spec=spec, election_bundle=election_bundle, genome=genome, episode_index=idx)
        archive.add({
            "genome": result["genome"],
            "score": result["score"],
            "completed_count": result["completed_count"],
            "failed_count": result["failed_count"],
            "mode_hint": result["mode_hint"],
            "workspace": result["workspace"],
        })
        episodes.append({
            "genome": result["genome"],
            "score": result["score"],
            "completed_count": result["completed_count"],
            "failed_count": result["failed_count"],
            "mode_hint": result["mode_hint"],
            "workspace": result["workspace"],
        })
        if champion is None or result["score"] > champion["score"]:
            champion = result

    workspace = Path(champion["workspace"])
    final_bundle = dict(champion["bundle"])
    final_bundle["arena"] = {"episodes": episodes, "champion": archive.champion()}
    final_bundle["plan_election"] = [{k: v for k, v in row.items() if k not in {"tasks", "payload"}} for row in election_bundle["election"]["rows"]]
    final_bundle["concept_script"] = CONCEPT_SCRIPT_TEXT
    final_bundle["prompt_tag_guide"] = PROMPT_TAG_GUIDE
    final_bundle["memory_advancement_guide"] = MEMORY_ADVANCEMENT_GUIDE_TEXT
    final_bundle["prompt_inventions"] = PROMPT_INVENTIONS
    emit_super_artifacts(workspace, final_bundle, [MentorEdge(**x) for x in champion["mentor_graph"]], champion["explanation_graph"], [OperatorAlert(**x) for x in champion["operator_alerts"]], archive)

    champion.update({
        "bundle": final_bundle,
        "arena": {"episodes": episodes, "champion": archive.champion()},
        "plan_election": final_bundle["plan_election"],
    })
    return champion


def main() -> Dict[str, Any]:
    return main_execution(
        spec=DEFAULT_SPEC,
        enable_starling=ENABLE_STARLING_LLM,
        max_steps=12,
        generated_task_budget=18,
    )


# Execution deferred to the upgraded runtime cell below.


In [ ]:

# @title 7b) Protocol-compiled task generation, braided memory, patch election, and upgraded main runtime
# This cell upgrades the runtime with stronger prompt envelopes, longer memory artifacts,
# and a concrete patch-election stage after the main execution pass.

import difflib

old_main_election = main_election
old_main_execution = main_execution

ADVANCED_PROMPT_TOKENS = {
    "system": "high-level runtime contract",
    "phase": "phase or stage fence",
    "action": "ordered action directive",
    "candidate": "candidate plan or task pack",
    "dependency": "dependency edge declaration",
    "teach": "cross-agent learning directive",
    "memory_braid": "long-horizon memory summary",
    "memory_capsule": "condensed replay artifact",
    "rarity_probe": "rare-condition instruction",
    "verify": "verification gate",
    "repair": "repair directive",
    "diff_request": "patch synthesis directive",
    "json_schema": "machine-readable response schema",
    "response": "outer envelope",
    "json": "strict JSON body",
}

@dataclass
class MemoryCapsule:
    capsule_id: str
    capsule_type: str
    score: float
    summary: str
    evidence: Dict[str, Any] = field(default_factory=dict)

class MemoryBraidEngine:
    def __init__(self, memory: MemoryStore):
        self.memory = memory

    def _capsule(self, capsule_id: str, capsule_type: str, traces: List[Dict[str, Any]], summary: str) -> MemoryCapsule:
        lane_counts = Counter(t.get("lane", "") or "none" for t in traces)
        event_counts = Counter(t.get("event_type", "") or "unknown" for t in traces)
        score = min(0.99, 0.18 + 0.07 * len(traces) + 0.03 * len(event_counts))
        return MemoryCapsule(
            capsule_id=capsule_id,
            capsule_type=capsule_type,
            score=round(score, 6),
            summary=summary,
            evidence={
                "trace_count": len(traces),
                "lane_counts": dict(lane_counts),
                "event_counts": dict(event_counts),
                "sample_task_ids": [t.get("task_id", "") for t in traces[:8]],
            },
        )

    def build_capsules(self, limit: int = 140) -> List[MemoryCapsule]:
        traces = self.memory.recent_traces(limit=limit)

        def sel(predicate):
            return [t for t in traces if predicate(t)]

        rare = sel(lambda t: bool((t.get("payload") or {}).get("rare_condition")) or "anomaly" in t.get("event_type", ""))
        colorsync = sel(lambda t: "colorsync" in t.get("event_type", "") or "broadcast" in t.get("event_type", "") or "teach" in t.get("event_type", ""))
        resets = sel(lambda t: "reset" in t.get("event_type", ""))
        repairs = sel(lambda t: "repair" in t.get("event_type", "") or "verify" in t.get("event_type", ""))
        election = sel(lambda t: "election" in t.get("event_type", "") or "champion" in t.get("event_type", ""))
        planning = sel(lambda t: t.get("phase") in {"plan", "task_generation", "prompt", "memory"})

        groups = [
            ("capsule-rare", "rare_condition", rare, "Replay rare-condition traces so future runs can detect palette shifts earlier."),
            ("capsule-colorsync", "colorsync", colorsync, "Track which palettes propagated across agents and whether the broadcast was stabilizing."),
            ("capsule-reset", "reset", resets, "Capture why the runtime paused, recollected, and resumed."),
            ("capsule-repair", "repair", repairs, "Store the verification->repair->reverify ladder so the next run can bootstrap from proven fixes."),
            ("capsule-election", "election", election, "Preserve plan-election evidence, champion selection rationale, and genome outcomes."),
            ("capsule-planning", "planning", planning, "Keep the project objective, lane map, and control-surface reasoning visible to later prompts."),
        ]

        capsules = []
        for capsule_id, capsule_type, group, summary in groups:
            if group:
                capsules.append(self._capsule(capsule_id, capsule_type, group, summary))

        if not capsules:
            capsules.append(self._capsule(
                "capsule-empty",
                "bootstrap",
                traces[:12],
                "No strong memory cluster found; bootstrap with the most recent execution traces.",
            ))
        return capsules

    def render_packet(self, limit: int = 140) -> str:
        traces = self.memory.recent_traces(limit=limit)
        event_counts = Counter(t.get("event_type", "") or "unknown" for t in traces)
        phase_counts = Counter(t.get("phase", "") or "unknown" for t in traces)
        capsules = self.build_capsules(limit=limit)
        parts = [
            f'[memory_braid trace_count="{len(traces)}" unique_events="{len(event_counts)}" unique_phases="{len(phase_counts)}"]',
            "EVENT_COUNTS: " + json.dumps(dict(event_counts), sort_keys=True),
            "PHASE_COUNTS: " + json.dumps(dict(phase_counts), sort_keys=True),
        ]
        for cap in capsules:
            parts.append(
                f'[memory_capsule id="{cap.capsule_id}" type="{cap.capsule_type}" score="{cap.score}"]\n'
                f'SUMMARY: {cap.summary}\n'
                f'EVIDENCE: {json.dumps(cap.evidence, sort_keys=True)}\n'
                f'[/memory_capsule]'
            )
        parts.append("[/memory_braid]")
        return "\n".join(parts)

class PromptProtocolCompiler:
    def __init__(self, concept_script_text: str):
        self.concept_script_text = concept_script_text

    def response_schema(self, max_tasks: int = 20) -> Dict[str, Any]:
        return {
            "type": "object",
            "required": ["candidate_id", "mode_hint", "ordered_actions", "tasks", "memory_writes", "patch_intents"],
            "properties": {
                "candidate_id": {"type": "string"},
                "mode_hint": {"type": "string", "enum": ["reasoning", "maintenance", "anomaly_detection", "repair"]},
                "ordered_actions": {
                    "type": "array",
                    "minItems": 10,
                    "items": {
                        "type": "object",
                        "required": ["step", "token", "summary"],
                        "properties": {
                            "step": {"type": "integer"},
                            "token": {"type": "string"},
                            "summary": {"type": "string"},
                        },
                    },
                },
                "tasks": {
                    "type": "array",
                    "maxItems": max_tasks,
                    "items": {
                        "type": "object",
                        "required": ["title", "lane", "description", "dependencies", "resources", "priority", "uncertainty", "rationale"],
                        "properties": {
                            "title": {"type": "string"},
                            "lane": {"type": "string"},
                            "description": {"type": "string"},
                            "dependencies": {"type": "array", "items": {"type": "string"}},
                            "resources": {"type": "array", "items": {"type": "string"}},
                            "priority": {"type": "number"},
                            "uncertainty": {"type": "number"},
                            "rationale": {"type": "string"},
                            "rare_condition": {"type": "boolean"},
                            "learning_value": {"type": "number"},
                            "color_hint": {"type": "object"},
                        },
                    },
                },
                "memory_writes": {"type": "array", "items": {"type": "string"}},
                "patch_intents": {"type": "array", "items": {"type": "string"}},
                "mentor_hypotheses": {"type": "array", "items": {"type": "string"}},
            },
        }

    def compile_task_generator_prompt(
        self,
        spec: str,
        state: PlannerState,
        memory_packet: str,
        concept_bank: Dict[str, Any],
        max_tasks: int = 20,
    ) -> str:
        ready = [
            {
                "task_id": t.task_id,
                "title": t.title,
                "lane": t.lane,
                "dependencies": t.dependencies,
                "resources": t.resources,
                "priority": t.priority,
                "uncertainty": t.uncertainty,
            }
            for t in state.ready_tasks()[:18]
        ]
        lane_counts = Counter(t.lane for t in state.tasks.values())
        resource_hotspots = Counter(res for t in state.tasks.values() for res in t.resources).most_common(16)
        schema_text = json.dumps(self.response_schema(max_tasks=max_tasks), indent=2)
        action_script = build_ordered_action_script(max_tasks=max_tasks)
        concept_excerpt = json.dumps({
            "blog_title": concept_bank.get("series_title", ""),
            "topic": concept_bank.get("topic", ""),
            "objective": concept_bank.get("objective", ""),
        }, indent=2)

        sections = [
            '[system role="starling_runtime_planner" style="strict_tagged_protocol"]',
            "You are a local-planner runtime that must be precise, responsive, and dependency-safe.",
            "Write with short action-focused sentences inside tags, then emit strict JSON inside [json].",
            "Never drift into generic prose. Favor executable task packs, concrete files, and lane-safe sequencing.",
            '[/system]',
            f'[phase name="interpret_objective"]\n{spec}\n[/phase]',
            f'[phase name="current_runtime_state"]\nREADY_TASKS={json.dumps(ready, indent=2)}\nLANE_COUNTS={json.dumps(dict(lane_counts), indent=2)}\nRESOURCE_HOTSPOTS={json.dumps(resource_hotspots, indent=2)}\n[/phase]',
            f'[phase name="concept_bank"]\n{concept_excerpt}\n[/phase]',
            f'[phase name="concept_script"]\n{self.concept_script_text}\n[/phase]',
            f'[phase name="memory"]\n{memory_packet}\n[/phase]',
            f'[phase name="ordered_actions"]\n{action_script}\n[/phase]',
            '[rarity_probe]\nAdd at least one rare-condition task and one anomaly-forecast task that can surface hidden failure modes.\n[/rarity_probe]',
            '[teach]\nAdd at least two tasks where agents learn from other agents via ColorSync palette observations, mentorship replay, or palette quarantine.\n[/teach]',
            '[verify]\nRequire contracts, verification gates, and repair readiness before claiming completion.\n[/verify]',
            '[repair]\nIf a task can fail, add rollback and reverify hooks in either the task list or the memory_writes section.\n[/repair]',
            '[diff_request]\nSuggest at least two concrete patch intents for files or contracts the runtime should produce after execution.\n[/diff_request]',
            f'[json_schema]\n{schema_text}\n[/json_schema]',
            '[response]\n[action step="1" token="interpret_objective"]Summarize the repository shape and minimal control surface.[/action]\n'
            '[action step="2" token="elect_lanes"]Name the lanes and why they must exist.[/action]\n'
            '[action step="3" token="mapf_control"]Add reservation, handoff, and conflict-avoidance tasks.[/action]\n'
            '[action step="4" token="chromatic_quantum"]Add chromatic palette and PennyLane-ready scoring tasks.[/action]\n'
            '[action step="5" token="memory_braid"]Add braid, replay, capsule, and calibration tasks.[/action]\n'
            '[action step="6" token="rarity_probe"]Add rare-condition, anomaly, and outlier probes.[/action]\n'
            '[action step="7" token="verify_repair"]Add verification, contract, repair, and rollback tasks.[/action]\n'
            '[action step="8" token="observability"]Add operator dashboards, artifacts, and runtime reports.[/action]\n'
            '[action step="9" token="colorsync_learning"]Add explicit cross-agent learning tasks.[/action]\n'
            '[action step="10" token="dependency_order"]Ensure the final task list is dependency-safe and execution-ready.[/action]\n'
            '[json]\n{\n  "candidate_id": "string",\n  "mode_hint": "reasoning|maintenance|anomaly_detection|repair",\n  "ordered_actions": [],\n  "tasks": [],\n  "memory_writes": [],\n  "patch_intents": [],\n  "mentor_hypotheses": []\n}\n[/json]\n[/response]',
        ]
        return "\n\n".join(sections)

@dataclass
class PatchProposal:
    patch_id: str
    target_path: str
    before: str
    after: str
    rationale: str
    tags: List[str] = field(default_factory=list)
    score: float = 0.0

class DiffScorer:
    @staticmethod
    def score(before: str, after: str, target_path: str, tags: List[str]) -> float:
        before_lines = before.splitlines()
        after_lines = after.splitlines()
        added = max(0, len(after_lines) - len(before_lines))
        changed = sum(1 for line in difflib.ndiff(before_lines, after_lines) if line.startswith("+ ") or line.startswith("- "))
        structural_bonus = 0.0
        if target_path.endswith(".py"):
            structural_bonus += 0.20
            structural_bonus += 0.10 * sum(tok in after for tok in ["def planner_mode", "def memory_capsule_count", "def patch_protocol"])
        if target_path.endswith(".md"):
            structural_bonus += 0.14
        if target_path.endswith(".json"):
            structural_bonus += 0.12
        if "memory" in target_path:
            structural_bonus += 0.10
        if "protocol" in target_path:
            structural_bonus += 0.08
        tag_bonus = min(0.25, 0.03 * len(tags))
        size_penalty = min(0.20, 0.003 * max(0, added - 50))
        return round(0.36 + structural_bonus + tag_bonus + min(0.18, 0.004 * changed) - size_penalty, 6)

class PatchSynthesizer:
    def __init__(self, workspace: Path):
        self.workspace = workspace

    def _read(self, rel: str) -> str:
        path = self.workspace / rel
        return path.read_text() if path.exists() else ""

    def propose(self, run_result: Dict[str, Any], memory_packet: str, prompt_text: str) -> List[PatchProposal]:
        proposals: List[PatchProposal] = []

        # 1) Runtime core extension
        core_rel = "src/runtime/core.py"
        core_before = self._read(core_rel)
        core_append = textwrap.dedent(f"""
        def planner_mode() -> str:
            return "{run_result.get('mode_hint', 'reasoning')}"

        def memory_capsule_count() -> int:
            return {len(run_result.get('memory_capsules', []))}

        def patch_protocol() -> str:
            return "protocol_compiled_v2"
        """).strip() + "\n"
        core_after = core_before
        if "def planner_mode" not in core_before:
            core_after = core_before.rstrip() + "\n\n" + core_append if core_before.strip() else core_append
        proposals.append(PatchProposal(
            patch_id="patch-runtime-core",
            target_path=core_rel,
            before=core_before,
            after=core_after,
            rationale="Extend the runtime core with mode, memory-capsule, and protocol helpers so the upgraded execution path becomes observable.",
            tags=["runtime", "protocol", "memory"],
        ))

        # 2) Smoke test extension
        test_rel = "tests/test_runtime_smoke.py"
        test_before = self._read(test_rel)
        test_append = textwrap.dedent("""
        from src.runtime.core import planner_mode, memory_capsule_count, patch_protocol

        def test_planner_mode_contract():
            assert planner_mode() in {"reasoning", "maintenance", "anomaly_detection", "repair"}
            assert isinstance(memory_capsule_count(), int)
            assert patch_protocol() == "protocol_compiled_v2"
        """).strip() + "\n"
        test_after = test_before
        if "test_planner_mode_contract" not in test_before:
            test_after = test_before.rstrip() + "\n\n" + test_append if test_before.strip() else test_append
        proposals.append(PatchProposal(
            patch_id="patch-runtime-tests",
            target_path=test_rel,
            before=test_before,
            after=test_after,
            rationale="Verify that the upgraded runtime exports planner mode and memory protocol markers.",
            tags=["tests", "verification", "protocol"],
        ))

        # 3) Main runtime protocol doc
        doc_rel = "docs/main_runtime_protocol.md"
        doc_before = self._read(doc_rel)
        doc_after = textwrap.dedent(f"""
        # Main Runtime Protocol v2

        This runtime now executes in the following order:

        1. bootstrap deterministic and generated task packs
        2. elect a champion plan
        3. execute stage-gated MAPF-safe tasks
        4. propagate ColorSync learning and rare-condition signals
        5. verify and repair the workspace
        6. braid memory traces into reusable capsules
        7. elect and apply patch proposals
        8. archive operator-visible artifacts

        ## Mode hint
        `{run_result.get("mode_hint", "reasoning")}`

        ## Prompt protocol
        The upgraded task generator uses tagged envelopes such as `[action]`, `[memory_braid]`,
        `[memory_capsule]`, `[verify]`, `[repair]`, and `[diff_request]`.

        ## Memory packet preview
        ```
        {memory_packet[:3500]}
        ```
        """).strip() + "\n"
        proposals.append(PatchProposal(
            patch_id="patch-runtime-doc",
            target_path=doc_rel,
            before=doc_before,
            after=doc_after,
            rationale="Explain the explicit main execution flow and the upgraded tagged prompt protocol.",
            tags=["docs", "protocol", "operator"],
        ))

        # 4) Memory braid snapshot
        braid_rel = "reports/memory_braid_snapshot.json"
        braid_before = self._read(braid_rel)
        braid_after = json.dumps({
            "memory_capsules": [asdict(x) for x in run_result.get("memory_capsules", [])],
            "prompt_token_count": sum(len(v) for v in ADVANCED_PROMPT_TOKENS.values()),
            "memory_packet_preview": memory_packet[:4000],
        }, indent=2)
        proposals.append(PatchProposal(
            patch_id="patch-memory-braid",
            target_path=braid_rel,
            before=braid_before,
            after=braid_after,
            rationale="Persist the braided memory capsules and packet preview as an operator-visible replay artifact.",
            tags=["memory", "reports", "replay"],
        ))

        # 5) Prompt protocol contract
        contract_rel = "contracts/prompt_protocol_v2.json"
        contract_before = self._read(contract_rel)
        contract_after = json.dumps({
            "tokens": ADVANCED_PROMPT_TOKENS,
            "required_sections": ["system", "phase", "memory_braid", "teach", "verify", "repair", "diff_request", "response", "json"],
            "response_schema_keys": ["candidate_id", "mode_hint", "ordered_actions", "tasks", "memory_writes", "patch_intents", "mentor_hypotheses"],
            "prompt_preview": prompt_text[:5000],
        }, indent=2)
        proposals.append(PatchProposal(
            patch_id="patch-prompt-contract",
            target_path=contract_rel,
            before=contract_before,
            after=contract_after,
            rationale="Make the upgraded prompt grammar explicit and machine-readable.",
            tags=["contracts", "prompt", "schema"],
        ))

        for proposal in proposals:
            proposal.score = DiffScorer.score(proposal.before, proposal.after, proposal.target_path, proposal.tags)
        return proposals

def unified_diff_for_patch(p: PatchProposal) -> str:
    return "\n".join(difflib.unified_diff(
        p.before.splitlines(),
        p.after.splitlines(),
        fromfile=f"a/{p.target_path}",
        tofile=f"b/{p.target_path}",
        lineterm=""
    ))

def elect_patch_set(proposals: List[PatchProposal], min_score: float = 0.48) -> Dict[str, Any]:
    ranked = sorted(proposals, key=lambda p: (-p.score, p.target_path))
    selected = [p for p in ranked if p.score >= min_score]
    return {
        "selected_patch_ids": [p.patch_id for p in selected],
        "selected_count": len(selected),
        "ranked": [
            {"patch_id": p.patch_id, "target_path": p.target_path, "score": p.score, "tags": p.tags, "rationale": p.rationale}
            for p in ranked
        ],
    }, selected

def apply_patch_set(workspace: Path, selected: List[PatchProposal]) -> List[str]:
    applied = []
    diff_dir = workspace / "reports" / "patch_diffs"
    diff_dir.mkdir(parents=True, exist_ok=True)
    for proposal in selected:
        path = workspace / proposal.target_path
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(proposal.after)
        (diff_dir / f"{proposal.patch_id}.diff").write_text(unified_diff_for_patch(proposal))
        applied.append(proposal.target_path)
    return applied

def write_memory_seed(memory: MemoryStore, run_result: Dict[str, Any]) -> None:
    memory.write_trace("execution", None, "execution_summary", {
        "mode_hint": run_result.get("mode_hint"),
        "score": run_result.get("score"),
        "completed_count": run_result.get("completed_count"),
        "failed_count": run_result.get("failed_count"),
    })
    for event in run_result.get("colorsync_events", [])[:24]:
        memory.write_trace("colorsync", None, "colorsync_broadcast", event if isinstance(event, dict) else {"note": str(event)})
    for event in run_result.get("reset_events", [])[:24]:
        memory.write_trace("reset", None, "reset_event", event if isinstance(event, dict) else {"note": str(event)})
    for event in run_result.get("teaching_log", [])[:24]:
        memory.write_trace("mentor", None, "teaching_event", event if isinstance(event, dict) else {"note": str(event)})
    for event in run_result.get("anomaly_log", [])[:24]:
        memory.write_trace("anomaly", None, "anomaly_event", event if isinstance(event, dict) else {"note": str(event)})
    for row in run_result.get("plan_election", [])[:12]:
        memory.write_trace("election", None, "plan_election_row", row if isinstance(row, dict) else {"note": str(row)})
    bundle = run_result.get("bundle", {})
    if bundle:
        memory.write_trace("bundle", None, "bundle_written", {
            "keys": sorted(bundle.keys()),
            "verification_after": bundle.get("verification_after", {}),
            "smoke_tests": bundle.get("smoke_tests", {}),
        })

def main_election(spec: str, enable_starling: bool = False, generated_task_budget: int = 20) -> Dict[str, Any]:
    # Wrapper keeps the stable election implementation, while adding protocol-compiled prompt artifacts.
    base = old_main_election(spec=spec, enable_starling=enable_starling, generated_task_budget=generated_task_budget)
    concept_bank = load_concept_bank()
    state = PlannerState(spec=spec)
    state.add_tasks(dedupe_tasks(deterministic_seed_tasks(spec) + build_advanced_runtime_tasks(start_index=880)))
    memory = MemoryStore(ADVANCED_WORKSPACE_ROOT.parent / "agentic_runtime_v15_election_memory.db")
    memory.write_trace("prompt", None, "election_bootstrap", {"generated_task_budget": generated_task_budget, "spec": spec[:300]})
    braid = MemoryBraidEngine(memory)
    compiler = PromptProtocolCompiler(CONCEPT_SCRIPT_TEXT)
    protocol_prompt = compiler.compile_task_generator_prompt(spec, state, braid.render_packet(limit=80), concept_bank, max_tasks=generated_task_budget)
    base["protocol_prompt"] = protocol_prompt
    base["prompt_protocol_contract"] = compiler.response_schema(max_tasks=generated_task_budget)
    return base

def main_execution(spec: str, enable_starling: bool = False, max_steps: int = 12, generated_task_budget: int = 20) -> Dict[str, Any]:
    base = old_main_execution(spec=spec, enable_starling=enable_starling, max_steps=max_steps, generated_task_budget=generated_task_budget)
    workspace = Path(base["workspace"])
    concept_bank = load_concept_bank()
    memory = MemoryStore(workspace.parent / f"{workspace.name}_v15_memory.db")
    write_memory_seed(memory, base)
    braid = MemoryBraidEngine(memory)
    memory_packet = braid.render_packet(limit=140)
    memory_capsules = braid.build_capsules(limit=140)

    # Build an upgraded protocol prompt against the current execution state.
    state = PlannerState(spec=spec)
    combined_tasks = dedupe_tasks(deterministic_seed_tasks(spec) + build_advanced_runtime_tasks(start_index=920))
    state.add_tasks(combined_tasks)
    state.completed.extend([t.get("task_id", "") for t in base.get("bundle", {}).get("tasks", []) if isinstance(t, dict) and t.get("status") == "completed"])
    compiler = PromptProtocolCompiler(CONCEPT_SCRIPT_TEXT)
    upgraded_prompt = compiler.compile_task_generator_prompt(spec, state, memory_packet, concept_bank, max_tasks=generated_task_budget + 2)

    # Persist prompt and memory artifacts before patches.
    (workspace / "docs").mkdir(parents=True, exist_ok=True)
    (workspace / "reports").mkdir(parents=True, exist_ok=True)
    (workspace / "contracts").mkdir(parents=True, exist_ok=True)
    (workspace / "docs/advanced_task_generator_prompt_v2.txt").write_text(upgraded_prompt)
    (workspace / "docs/memory_braid_packet.txt").write_text(memory_packet)
    (workspace / "contracts/advanced_task_generator_schema_v2.json").write_text(json.dumps(compiler.response_schema(max_tasks=generated_task_budget + 2), indent=2))
    (workspace / "reports/memory_capsules.json").write_text(json.dumps([asdict(x) for x in memory_capsules], indent=2))

    synthesizer = PatchSynthesizer(workspace)
    proposals = synthesizer.propose({**base, "memory_capsules": memory_capsules}, memory_packet, upgraded_prompt)
    patch_election, selected = elect_patch_set(proposals, min_score=0.47)
    applied_paths = apply_patch_set(workspace, selected)

    verification_upgrade = verify_workspace(workspace)
    smoke_upgrade = run_smoke_tests(workspace)

    patch_bundle = {
        "patch_election": patch_election,
        "applied_paths": applied_paths,
        "verification_upgrade": verification_upgrade,
        "smoke_upgrade": smoke_upgrade,
        "memory_capsules": [asdict(x) for x in memory_capsules],
        "prompt_tokens": ADVANCED_PROMPT_TOKENS,
    }
    (workspace / "reports/patch_election.json").write_text(json.dumps(patch_bundle, indent=2))
    (workspace / "docs/prompt_protocol_v2.md").write_text(textwrap.dedent(f"""
    # Prompt Protocol v2

    The runtime now uses a tagged control grammar to help Starling/Sterling-style local inference stay coherent.

    ## Control tokens
    {json.dumps(ADVANCED_PROMPT_TOKENS, indent=2)}

    ## Concept script preview
    ```
    {CONCEPT_SCRIPT_TEXT[:5000]}
    ```

    ## Ordered planning procedure
    {json.dumps(ORDERED_PLANNING_PROCEDURE, indent=2)}
    """).strip() + "\n")

    advanced_score = round(
        float(base.get("score", 0.0))
        + 0.02 * len(memory_capsules)
        + 0.015 * len(applied_paths)
        + (0.05 if smoke_upgrade.get("returncode", 1) == 0 else -0.04)
        + (0.04 if verification_upgrade.get("syntax_error_count", 1) == 0 else -0.04),
        6,
    )

    base.update({
        "memory_packet": memory_packet,
        "memory_capsules": memory_capsules,
        "advanced_prompt_text": upgraded_prompt,
        "patch_bundle": patch_bundle,
        "verification_upgrade": verification_upgrade,
        "smoke_tests_upgrade": smoke_upgrade,
        "score": advanced_score,
    })
    return base

def main() -> Dict[str, Any]:
    return main_execution(
        spec=DEFAULT_SPEC,
        enable_starling=ENABLE_STARLING_LLM,
        max_steps=13,
        generated_task_budget=20,
    )

run_result = main()
summary = {
    "workspace": run_result["workspace"],
    "backend": run_result["backend"],
    "mode_hint": run_result["mode_hint"],
    "score": run_result["score"],
    "completed_count": run_result["completed_count"],
    "failed_count": run_result["failed_count"],
    "colorsync_events": len(run_result["colorsync_events"]),
    "reset_events": len(run_result["reset_events"]),
    "teaching_events": len(run_result["teaching_log"]),
    "memory_capsules": len(run_result["memory_capsules"]),
    "patches_applied": len(run_result["patch_bundle"]["applied_paths"]),
    "smoke_tests_returncode": run_result["smoke_tests_upgrade"]["returncode"],
}
print(json.dumps(summary, indent=2))


In [ ]:

# @title 8) Inspect upgraded artifacts, prompt protocol, memory braid, patch election, and explanation graph
workspace_path = Path(run_result["workspace"])
artifact_paths = sorted(str(p.relative_to(workspace_path)) for p in workspace_path.rglob("*") if p.is_file())
print(json.dumps({
    "workspace": str(workspace_path),
    "artifact_count": len(artifact_paths),
    "sample_artifacts": artifact_paths[:40],
    "mode_hint": run_result["mode_hint"],
    "score": run_result["score"],
    "completed_count": run_result["completed_count"],
    "failed_count": run_result["failed_count"],
    "colorsync_events": len(run_result["colorsync_events"]),
    "reset_events": len(run_result["reset_events"]),
    "teaching_events": len(run_result["teaching_log"]),
    "memory_capsules": len(run_result["memory_capsules"]),
    "patches_applied": len(run_result["patch_bundle"]["applied_paths"]),
    "operator_alerts": len(run_result["operator_alerts"]),
}, indent=2))

print("\n--- memory braid packet preview ---\n")
print(run_result["memory_packet"][:3200])

print("\n--- memory capsules ---\n")
for row in run_result["memory_capsules"][:8]:
    print(json.dumps(asdict(row), indent=2) if hasattr(row, "__dataclass_fields__") else json.dumps(row, indent=2))

print("\n--- patch election ---\n")
print(json.dumps(run_result["patch_bundle"]["patch_election"], indent=2))

print("\n--- protocol prompt preview ---\n")
print(run_result["advanced_prompt_text"][:3800])

print("\n--- explanation graph summary ---\n")
print(json.dumps({
    "node_count": len(run_result["explanation_graph"]["nodes"]),
    "edge_count": len(run_result["explanation_graph"]["edges"]),
    "mentor_edges": len(run_result["mentor_graph"]),
    "operator_alerts": len(run_result["operator_alerts"]),
}, indent=2))
